# model

> Build a segmentation model (any `segmentation_models_pytorch` architecture + encoder) with configurable `in_channels` and `num_classes`, plus helpers for differential learning rates and encoder freezing.

We use `segmentation_models_pytorch`, which adapts the pretrained encoder's first convolution to any `in_channels` and lets us set `classes` to 1 (binary) or N (multi-class) directly.


In [ ]:
#| default_exp model

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections.abc import Sequence
from typing import List, Optional, Set, Union
import math
from functools import partial
import warnings

import torch
import torch.nn as nn


In [ ]:
#| export
import segmentation_models_pytorch as smp

## Build the model

`architecture` selects the smp segmentation head (`unet`, `fpn`, `deeplabv3plus`, …). `arch` / `encoder` selects any smp encoder backbone (`mobilenet_v2`, `resnet34`, `efficientnet-b3`, …).

`num_classes == 1` produces a single logit channel for sigmoid; `num_classes > 1` produces one channel per class for softmax. `activation=None` keeps the head returning raw logits, which is what our losses and metrics expect.


### Scratch — local ImageNet encoder weights

The Hugging Face hub cache path
`~/.cache/huggingface/hub/models--smp-hub--mobilenet_v2.imagenet` is a
**directory** (`blobs/` + `snapshots/`), not a file. `torch.load` needs a file.

Two workable paths:

1. **Online / cache already warm:** `encoder_weights='imagenet'` — SMP finds the hub snapshot itself.
2. **Portable for later (air-gapped):** save an encoder-only `.pth`, then load that file
   (same idea as `download_encoder_weights` + `get_model(weights_path=...)` below).


In [ ]:
#| eval: false
# experiment: resolve a real weight *file* (or None → let SMP use imagenet)
from pathlib import Path

HF_MOBILENET = Path.home() / '.cache/huggingface/hub/models--smp-hub--mobilenet_v2.imagenet'
# portable encoder dump you can copy to HPC later — change if you prefer another folder
PORTABLE_PTH = Path('/tmp/segmentation_tools_pt_weights/mobilenet_v2_encoder.pth')

def resolve_encoder_weight_file(hf_repo_dir: Path = HF_MOBILENET, portable: Path = PORTABLE_PTH):
    "Prefer a portable .pth; else the hub snapshot model.safetensors; else None."
    if portable.is_file():
        return portable
    if hf_repo_dir.is_dir():
        # newest snapshot's model.safetensors (HF layout)
        snaps = sorted((hf_repo_dir / 'snapshots').glob('*/model.safetensors'))
        if snaps:
            return snaps[-1]
    return None

weights_file = resolve_encoder_weight_file()
print('weights_file =', weights_file)
print('is_file =', weights_file.is_file() if weights_file else None)


In [ ]:
#| eval: false
# verify: never hand a directory to torch.load
assert weights_file is None or weights_file.is_file(), f'not a file: {weights_file}'
assert not HF_MOBILENET.is_file(), 'hub root must be a directory'
print('step ok: path is a file or None')


In [ ]:
#| eval: false
# experiment: build Unet — SMP imagenet if no portable file yet; else load encoder from file
in_channels, num_classes = 3, 1
pretrained = True

if weights_file is not None and weights_file.suffix == '.pth':
    model = smp.Unet(
        encoder_name='mobilenet_v2', encoder_weights=None,
        in_channels=in_channels, classes=num_classes, activation=None,
    )
    state = torch.load(weights_file, map_location='cpu', weights_only=False)
    missing, unexpected = model.encoder.load_state_dict(state, strict=False)
    print(f'loaded encoder .pth from {weights_file}')
    print('missing', len(missing), 'unexpected', len(unexpected))
elif weights_file is not None and weights_file.suffix == '.safetensors':
    # Hub snapshot is safetensors; easiest is still SMP's imagenet loader (uses this cache).
    # Optional: write a portable .pth for air-gap copy (next cell).
    model = smp.Unet(
        encoder_name='mobilenet_v2', encoder_weights='imagenet' if pretrained else None,
        in_channels=in_channels, classes=num_classes, activation=None,
    )
    print(f'SMP imagenet load (hub cache at {HF_MOBILENET.name})')
else:
    model = smp.Unet(
        encoder_name='mobilenet_v2', encoder_weights='imagenet' if pretrained else None,
        in_channels=in_channels, classes=num_classes, activation=None,
    )
    print('SMP imagenet download/cache')

y = model(torch.randn(1, in_channels, 64, 64))
print('out', tuple(y.shape))


In [ ]:
#| eval: false
# verify: forward shape
assert tuple(y.shape) == (1, num_classes, 64, 64)
print('forward ok')


In [ ]:
#| eval: false
# experiment: save portable encoder .pth for later / HPC (no hub directory needed)
PORTABLE_PTH.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.encoder.state_dict(), PORTABLE_PTH)
print(f'saved encoder → {PORTABLE_PTH} ({PORTABLE_PTH.stat().st_size/1e6:.1f} MB)')


In [ ]:
#| eval: false
# verify: reload from portable .pth only (simulates air-gapped load)
m2 = smp.Unet(
    encoder_name='mobilenet_v2', encoder_weights=None,
    in_channels=in_channels, classes=num_classes, activation=None,
)
state2 = torch.load(PORTABLE_PTH, map_location='cpu', weights_only=False)
m2.encoder.load_state_dict(state2, strict=False)
assert tuple(m2(torch.randn(1, in_channels, 64, 64)).shape) == (1, num_classes, 64, 64)
print('portable reload ok — use this path later with get_model(weights_path=...)')
print(PORTABLE_PTH)


In [ ]:
import torch; print(torch.__version__)

In [ ]:
#| export
SMP_ARCHITECTURES = {
    'unet': smp.Unet,
    'unetplusplus': smp.UnetPlusPlus,
    'manet': smp.MAnet,
    'linknet': smp.Linknet,
    'fpn': smp.FPN,
    'pspnet': smp.PSPNet,
    'pan': smp.PAN,
    'deeplabv3': smp.DeepLabV3,
    'deeplabv3plus': smp.DeepLabV3Plus,
    'upernet': smp.UPerNet,
}


def list_smp_architectures() -> List[str]:
    """Segmentation head names accepted by `get_model(architecture=...)`."""
    return sorted(SMP_ARCHITECTURES.keys())


def list_smp_encoders() -> List[str]:
    """Encoder / backbone names from segmentation-models-pytorch (includes registered `dinov3_*`)."""
    return sorted(smp.encoders.get_encoder_names())


def _normalize_architecture(name: str) -> str:
    key = name.lower().replace('-', '').replace('_', '')
    aliases = {'unetplusplus': 'unetplusplus', 'deeplabv3plus': 'deeplabv3plus'}
    return aliases.get(key, key)


def _load_local_encoder_weights(model: nn.Module, weights_path: str, device: str) -> None:
    state = torch.load(weights_path, map_location=device, weights_only=False)
    model_state = model.encoder.state_dict()
    compatible = {
        k: v for k, v in state.items()
        if k in model_state and v.shape == model_state[k].shape
    }
    skipped = [
        k for k in state
        if k in model_state and state[k].shape != model_state[k].shape
    ]
    if skipped:
        print(
            f'WARNING: {len(skipped)} key(s) skipped from {weights_path} due to shape mismatch '
            f'(checkpoint in_channels ≠ model in_channels): {skipped}'
        )
    res = model.encoder.load_state_dict(compatible, strict=False)
    print(f'Loaded encoder from {weights_path} ({len(compatible)}/{len(model_state)} keys matched)')
    if res.missing_keys:
        print(f'  missing keys: {len(res.missing_keys)} (e.g. {res.missing_keys[0]})')
    if res.unexpected_keys:
        print(f'  unexpected keys: {len(res.unexpected_keys)} (e.g. {res.unexpected_keys[0]})')


def get_model(
    arch: str = 'mobilenet_v2',   # encoder name — alias kept for Hydra `model.arch`
    in_channels: int = 3,
    num_classes: int = 1,         # 1 = binary, >1 = multi-class
    pretrained: bool = True,      # ImageNet weights (ignored when weights_path is set)
    weights_path: Optional[str] = None,  # local encoder .pth from download_encoder_weights
    device: str = 'cpu',
    architecture: str = 'unet',   # smp head: unet | fpn | deeplabv3plus | ...
    encoder: Optional[str] = None,  # explicit encoder name; overrides `arch` when set
) -> nn.Module:
    """Build an smp model returning raw logits `(B, num_classes, H, W)`.

    Pass `weights_path` on air-gapped HPC instead of downloading ImageNet weights.
    Use `list_smp_architectures()` and `list_smp_encoders()` to discover valid names.
    `encoder='dinov3_vit_small'` / `'dinov3_vit_base'` build a DPT-reassembled DINOv3
    ViT encoder (see the DINOv3 section below) instead of a CNN backbone.

    Example:
        >>> m = get_model('resnet34', architecture='fpn', in_channels=3, num_classes=1, pretrained=False)
        >>> tuple(m(torch.randn(1, 3, 64, 64)).shape)
        (1, 1, 64, 64)
    """
    encoder_name = encoder if encoder is not None else arch
    arch_key = _normalize_architecture(architecture)
    if arch_key not in SMP_ARCHITECTURES:
        raise ValueError(
            f"Unknown architecture {architecture!r}. "
            f"Choose one of: {', '.join(list_smp_architectures())}"
        )
    valid_encoders = smp.encoders.get_encoder_names()
    if encoder_name not in valid_encoders:
        raise ValueError(
            f"Unknown encoder {encoder_name!r}. "
            f"Use list_smp_encoders() — {len(valid_encoders)} encoders available."
        )
    use_imagenet = pretrained and weights_path is None
    model_cls = SMP_ARCHITECTURES[arch_key]
    if encoder_name.startswith('dinov3_'):
        # DINOv3 weights are downloaded inside Dinov3ReassembleEncoder.__init__ via
        # timm's own hub, not smp's encoder_weights='imagenet' state-dict loader (which
        # would `load_state_dict` the *whole* encoder, including our reassemble layers,
        # against a checkpoint that only has the plain ViT backbone -> missing keys).
        # Thread `pretrained` through the registry's construction params instead.
        smp.encoders.encoders[encoder_name]['params']['pretrained'] = use_imagenet
        encoder_weights = None
    else:
        encoder_weights = 'imagenet' if use_imagenet else None
    model = model_cls(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )
    if weights_path is not None:
        _load_local_encoder_weights(model, weights_path, device)
    return model


In [ ]:
# test: output channels follow num_classes for binary and multi-class
m1 = get_model('mobilenet_v2', in_channels=2, num_classes=1, pretrained=False)
assert tuple(m1(torch.randn(1, 2, 64, 64)).shape) == (1, 1, 64, 64)
m3 = get_model('mobilenet_v2', in_channels=4, num_classes=3, pretrained=False)
assert tuple(m3(torch.randn(1, 4, 64, 64)).shape) == (1, 3, 64, 64)
m_fpn = get_model('resnet18', architecture='fpn', in_channels=3, num_classes=1, pretrained=False)
assert tuple(m_fpn(torch.randn(1, 3, 64, 64)).shape) == (1, 1, 64, 64)
assert 'resnet34' in list_smp_encoders()
assert 'deeplabv3plus' in list_smp_architectures()


In [ ]:
#| hide
#| eval: false
from pathlib import Path
weights_path = Path('/tmp/segmentation_tools_pt_weights/mobilenet_v2_encoder.pth')
print(f'Weights path exists: {weights_path.exists()} → {weights_path}')
m1 = get_model(
    'mobilenet_v2',
    in_channels=3,
    num_classes=1,
    pretrained=True,
    weights_path=str(weights_path) if weights_path.is_file() else None,
)
assert tuple(m1(torch.randn(1, 3, 64, 64)).shape) == (1, 1, 64, 64)


In [ ]:
#| eval: false
# (assert moved into the get_model scratch cell above)


## Offline / air-gapped HPC usage

On an HPC without internet, `encoder_weights='imagenet'` will raise a download error.  
The workflow is:

| Step | Where | Command |
|------|-------|---------|
| 1 | Any machine **with** internet | `download_encoder_weights('resnet34', '/tmp/resnet34.pth')` |
| 2 | Copy the `.pth` file to HPC | `scp /tmp/resnet34.pth hpc:/data/weights/` |
| 3 | HPC (no internet) | `get_model('resnet34', weights_path='/data/weights/resnet34.pth')` |

`download_encoder_weights` lets smp download + cache the weights normally, then saves **just the encoder state-dict** to a portable file.  
`get_model(weights_path=...)` builds the model with `encoder_weights=None` and loads that file — no network call is made.

In [ ]:
#| export
def download_encoder_weights(
    encoder_name: str,   # smp encoder name  (e.g. 'resnet34', 'efficientnet-b0')
    save_path: str,      # local .pth path to write   (e.g. '/data/weights/resnet34.pth')
) -> None:
    """Download ImageNet encoder weights to a portable local file.

    Run **once** on any machine that has internet access, copy the file to your
    air-gapped HPC, then use `get_model(weights_path=...)`.

    Internally builds a full smp.Unet to trigger the normal smp/timm download,
    then extracts and saves only the encoder state-dict — so the file is small
    (~20-100 MB) and independent of your `in_channels` / `num_classes` settings.

    Example:
        >>> import tempfile, os
        >>> p = os.path.join(tempfile.mkdtemp(), 'enc.pth')
        >>> download_encoder_weights('mobilenet_v2', p)   # doctest: +SKIP
        >>> os.path.exists(p)                              # doctest: +SKIP
        True
    """
    from pathlib import Path
    tmp = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                   in_channels=3, classes=1, activation=None)
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(tmp.encoder.state_dict(), save_path)
    n = len(tmp.encoder.state_dict())
    size_mb = Path(save_path).stat().st_size / 1e6
    print(f'Saved {encoder_name} encoder  ({n} tensors, {size_mb:.1f} MB)  →  {save_path}')
    del tmp


In [ ]:
#| eval: false
# What encoders are available in segmentation_models_pytorch (smp)?
import segmentation_models_pytorch as smp

# List all available encoders (common in stable SMP releases)
print("Available SMP encoders:")
for name in sorted(smp.encoders.get_encoder_names()):
    print("-", name)
    encoder_name = name
    try:
        download_encoder_weights(encoder_name, f'x:/Models/smp_models/{encoder_name}.pth')
    except Exception as e:
        print(f"Error downloading {encoder_name}: {e}")
    


In [ ]:
#| eval: false

# Example usage: pick one and download its weights
encoder_name = 'resnet18'  # or choose any from the printed list
download_encoder_weights(encoder_name, f'x:/Models/smp_models/{encoder_name}.pth')

## Differential learning rates and freezing

The pretrained encoder needs a gentler learning rate than the randomly-initialised decoder. Freezing the encoder for a few warm-up epochs lets the decoder catch up before fine-tuning the whole network.

In [42]:
#| export
def get_param_groups(
    model: nn.Module,        # an smp model (or any model; falls back to one group if no .encoder)
    encoder_lr: float = 1e-4,  # LR for the pretrained encoder
    decoder_lr: float = 1e-3,  # LR for the decoder + head
) -> List[dict]:
    "Two parameter groups so encoder and decoder train at different rates."
    if not hasattr(model, 'encoder'):
        return [{'params': list(model.parameters()), 'lr': decoder_lr}]
    enc = list(model.encoder.parameters())
    if hasattr(model, 'decoder_side_parameters'):  # a model with modules beyond encoder/decoder/head lists them itself
        dec = list(model.decoder_side_parameters())
    else:
        dec = list(model.decoder.parameters()) + list(model.segmentation_head.parameters())
    return [{'params': enc, 'lr': encoder_lr}, {'params': dec, 'lr': decoder_lr}]

def freeze_encoder(model: nn.Module) -> None:
    "Freeze encoder weights for a decoder warm-up phase. No-op if model has no .encoder."
    if not hasattr(model, 'encoder'):
        return
    for p in model.encoder.parameters():
        p.requires_grad = False

def unfreeze_encoder(model: nn.Module) -> None:
    "Unfreeze the encoder after warm-up. No-op if model has no .encoder."
    if not hasattr(model, 'encoder'):
        return
    for p in model.encoder.parameters():
        p.requires_grad = True

## DINOv3 encoder-decoder

`list_smp_encoders()` only covers CNN-style backbones — there is no DINOv3 entry, because
DINOv3 is a plain Vision Transformer: `timm`'s `vit_*_patch16_dinov3` models emit a single
token grid at stride 16, not the stride `[2,4,8,16,32]` feature pyramid smp's `Unet`/`FPN`
decoders expect for their skip connections.

To use a DINOv3 backbone with those decoders anyway, we tap 4 evenly-spaced transformer
blocks (`timm`'s `forward_intermediates`) and reassemble each one to a different stride via
up/down-sampling convs — the same trick the DPT paper (Ranftl et al., 2021) uses to turn a
ViT into a dense-prediction backbone. ViT has no natural high-resolution feature, so the
missing stride-2 skip is synthesized by further upsampling the stride-4 map; this gives the
decoder a shallow skip for sharp boundaries instead of starting at stride 4.

The result is registered into `smp.encoders.encoders` under `dinov3_vit_small` /
`dinov3_vit_base`, so it plugs into the existing `get_model(arch=..., architecture='unet')`
call — no new API, no changes to `get_param_groups` / `freeze_encoder` / Hydra configs.

Pretrained DINOv3 weights come from `timm`'s own hub download (`pretrained=True` at
construction), not smp's `encoder_weights='imagenet'` path — that path expects to
`load_state_dict` a full encoder checkpoint with no extra reassemble layers, which would
raise missing-key errors here. `get_model()` accounts for this below.


In [37]:
#| export
import timm
from segmentation_models_pytorch.encoders._base import EncoderMixin


In [38]:
#| export

class _Reassemble(nn.Module):
    "Project a ViT token feature map to `out_ch` channels, then resize to the target stride."
    def __init__(self, in_ch: int, out_ch: int, scale: float):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        if scale > 1:
            self.resize = nn.Upsample(scale_factor=scale, mode='bilinear', align_corners=False)
        elif scale < 1:
            self.resize = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=round(1 / scale), padding=1)
        else:
            self.resize = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.resize(self.proj(x))



In [39]:
#| export
class Dinov3ReassembleEncoder(nn.Module, EncoderMixin):
    """DINOv3 ViT backbone + DPT-style reassemble, exposed as an smp `EncoderMixin`
    so it can be used as `get_model(arch='dinov3_vit_small', architecture='unet', ...)`.

    Forward returns `[x, stride2, stride4, stride8, stride16, stride32]` — 6 tensors for
    `depth=5`, matching what a CNN encoder (e.g. resnet34) would return.
    """

    def __init__(
        self,
        model_name: str = 'vit_small_patch16_dinov3',
        depth: int = 5,
        output_stride: int = 32,
        pretrained: bool = True,
        widths=(64, 96, 192, 384, 384),  # channel widths for stride 2,4,8,16,32
    ):
        super().__init__()
        self._depth = depth
        self._in_channels = 3
        self._output_stride = output_stride

        # Always built for 3-channel input; `set_in_channels()` (called by smp's
        # `get_encoder()` right after construction) adapts the patch-embed conv for
        # any other `in_channels`, reusing the same weight-averaging trick smp uses
        # for every other encoder.
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, dynamic_img_size=True,
        )
        embed_dim = self.backbone.embed_dim

        w2, w4, w8, w16, w32 = widths
        self.reassemble = nn.ModuleList([
            _Reassemble(embed_dim, w4, scale=4),    # earliest tap  -> stride 4
            _Reassemble(embed_dim, w8, scale=2),    # ..            -> stride 8
            _Reassemble(embed_dim, w16, scale=1),   # ..            -> stride 16
            _Reassemble(embed_dim, w32, scale=0.5), # last tap      -> stride 32
        ])
        # ViT has no native high-res feature -- synthesize stride 2 by upsampling
        # the stride-4 map instead of leaving the decoder's shallowest skip empty.
        self.stem_up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(w4, w2, kernel_size=3, padding=1),
        )
        self._out_channels = (3, w2, w4, w8, w16, w32)

    def set_in_channels(self, in_channels: int, pretrained: bool = True) -> None:
        "Adapt the patch-embed conv in place; `in_chans` can't be set after `timm.create_model`."
        if in_channels == 3:
            return
        from segmentation_models_pytorch.encoders._utils import patch_first_conv
        self._in_channels = in_channels
        self._out_channels = (in_channels,) + self._out_channels[1:]
        patch_first_conv(self.backbone, new_in_channels=in_channels, pretrained=pretrained)

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        feats = self.backbone.forward_intermediates(
            x, indices=4, output_fmt='NCHW', intermediates_only=True,
        )
        stride4, stride8, stride16, stride32 = [r(f) for r, f in zip(self.reassemble, feats)]
        stride2 = self.stem_up(stride4)
        return [x, stride2, stride4, stride8, stride16, stride32]



In [40]:
#| export
def _register_dinov3_encoders() -> None:
    "Register the DINOv3 encoders into smp so `list_smp_encoders()` and `get_model()` see them."
    for name, model_name in [
        ('dinov3_vit_small', 'vit_small_patch16_dinov3'),
        ('dinov3_vit_base', 'vit_base_patch16_dinov3'),
    ]:
        smp.encoders.encoders[name] = {
            'encoder': Dinov3ReassembleEncoder,
            'pretrained_settings': {},  # unused -- pretrained loading happens in __init__ via timm's hub
            'params': {'model_name': model_name, 'pretrained': True},
        }

_register_dinov3_encoders()


In [43]:
# test: DINOv3 encoder is registered and pluggable through the normal get_model() call
assert {'dinov3_vit_small', 'dinov3_vit_base'} <= set(list_smp_encoders())

# small crop, no download -- pretrained=False just randomly inits the timm backbone
m_dino = get_model('dinov3_vit_small', architecture='unet', in_channels=3, num_classes=1, pretrained=False)
assert tuple(m_dino(torch.randn(1, 3, 64, 64)).shape) == (1, 1, 64, 64)

# single-channel input (grayscale X-ray, this project's common case) via set_in_channels
m_dino_gray = get_model('dinov3_vit_small', architecture='unet', in_channels=1, num_classes=1, pretrained=False)
assert tuple(m_dino_gray(torch.randn(1, 1, 64, 64)).shape) == (1, 1, 64, 64)
assert m_dino_gray.encoder.backbone.patch_embed.proj.in_channels == 1

# differential LR / freeze-unfreeze already work unchanged (encoder/decoder/segmentation_head present)
groups = get_param_groups(m_dino, encoder_lr=1e-5, decoder_lr=1e-3)
assert len(groups) == 2 and groups[0]['lr'] < groups[1]['lr']
freeze_encoder(m_dino)
assert all(not p.requires_grad for p in m_dino.encoder.parameters())
unfreeze_encoder(m_dino)
assert all(p.requires_grad for p in m_dino.encoder.parameters())


## Curated encoders and aliases

Problem: ConvNeXt-Tiny and EfficientNetV2-S already build via raw SMP `tu-*`
(`tu-convnext_tiny`, `tu-tf_efficientnetv2_s`), but our early `get_model` only
accepts names from `smp.encoders.get_encoder_names()`. Hydra presets and agents
cannot switch cleanly, and friendly names like `convnext_tiny` hard-error.

What we want, in order:

1. A small alias map: friendly name → curated `tu-*`.
2. Prove a curated `tu-*` forward pass works (synthetic tensor, no download).
3. Decide the policy: curated = silent; other `tu-*` = warn once; unknown non-`tu-*` = error.
4. Extend `list_smp_encoders()` so discovery shows curated aliases + raw curated `tu-*`.
5. Wire that policy into `get_model` / `download_encoder_weights`.
6. Confirm Hydra `configs/model/` packages compose (library MobileNet; multilabel DINOv3-S).


### Step 1 — alias map

Friendly Hydra names should resolve to the SMP `tu-*` strings we already know work.


In [44]:
# experiment: alias map (not exported yet)
aliases = {
    'convnext_tiny': 'tu-convnext_tiny',
    'efficientnetv2_s': 'tu-tf_efficientnetv2_s',
}
for friendly, raw in aliases.items():
    print(f'{friendly} -> {raw}')


convnext_tiny -> tu-convnext_tiny
efficientnetv2_s -> tu-tf_efficientnetv2_s


In [45]:
# verify: every friendly name maps to a tu-* string
assert aliases['convnext_tiny'].startswith('tu-')
assert aliases['efficientnetv2_s'].startswith('tu-')
assert set(aliases) == {'convnext_tiny', 'efficientnetv2_s'}
print('step 1 ok')


step 1 ok


### Step 2 — curated `tu-*` builds

SMP accepts these even when they are missing from `get_encoder_names()`. Check a
tiny forward pass with `pretrained=False` (no download).


In [46]:
# experiment: raw curated tu-* through smp.Unet
x = torch.randn(1, 1, 64, 64)
for enc in aliases.values():
    m = smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=1, classes=1, activation=None)
    y = m(x)
    print(enc, tuple(y.shape))


tu-convnext_tiny (1, 1, 64, 64)
tu-tf_efficientnetv2_s (1, 1, 64, 64)


In [47]:
# verify: output spatial size matches input; one logit channel
x = torch.randn(1, 1, 64, 64)
for enc in aliases.values():
    m = smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=1, classes=1, activation=None)
    assert tuple(m(x).shape) == (1, 1, 64, 64), enc
print('step 2 ok')


step 2 ok


### Step 3 — allowlist policy

- Name in SMP registry or curated `tu-*` → accept.
- Other `tu-*` → `UserWarning` once, then accept (experimental).
- Anything else → `ValueError`.


In [48]:
# experiment: policy sketched inline (not exported yet)
import warnings

curated_tu = set(aliases.values())
_warned = set()

def check_encoder(name: str) -> None:
    registry = set(smp.encoders.get_encoder_names())
    if name in registry or name in curated_tu:
        return
    if name.startswith('tu-'):
        if name not in _warned:
            warnings.warn(f'experimental tu-* encoder: {name}', UserWarning)
            _warned.add(name)
        return
    raise ValueError(f'Unknown encoder {name!r}')

print('check_encoder ready; curated_tu =', sorted(curated_tu))


check_encoder ready; curated_tu = ['tu-convnext_tiny', 'tu-tf_efficientnetv2_s']


In [49]:
# verify: allowlist policy (curated silent; experimental warn-once; unknown errors)
import warnings

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    check_encoder('tu-convnext_tiny')
    assert not w, w

with warnings.catch_warnings(record=True) as w1:
    warnings.simplefilter('always')
    check_encoder('tu-mobilenetv3_small_100')
    assert any('experimental' in str(x.message) for x in w1), w1
with warnings.catch_warnings(record=True) as w2:
    warnings.simplefilter('always')
    check_encoder('tu-mobilenetv3_small_100')
    assert not [x for x in w2 if 'experimental' in str(x.message)], w2

try:
    check_encoder('not_a_real_encoder_xyz')
    assert False, 'expected ValueError'
except ValueError as e:
    assert 'not_a_real_encoder_xyz' in str(e)
print('step 3 ok')


step 3 ok


### Step 4 — discovery list

`list_smp_encoders()` should advertise curated aliases and raw curated `tu-*`
alongside the SMP registry (including `dinov3_*` after registration above).
Experimental `tu-*` stay buildable but are not listed.


In [50]:
# experiment: discovery = registry ∪ aliases ∪ curated tu-*
discovered = set(smp.encoders.get_encoder_names())
discovered.update(aliases.keys())
discovered.update(aliases.values())
print('convnext_tiny' in discovered, 'tu-convnext_tiny' in discovered)
print('dinov3_vit_small' in discovered, 'mobilenet_v2' in discovered)
print('tu-mobilenetv3_small_100' in discovered)  # experimental: should stay False


True True
True True
False


In [51]:
# verify: curated advertised; experimental not listed
discovered = set(smp.encoders.get_encoder_names()) | set(aliases) | set(aliases.values())
assert {'convnext_tiny', 'efficientnetv2_s', 'tu-convnext_tiny', 'tu-tf_efficientnetv2_s'} <= discovered
assert {'mobilenet_v2', 'dinov3_vit_small'} <= discovered
assert 'tu-mobilenetv3_small_100' not in discovered
print('step 4 ok')


step 4 ok


### Promote — export the working pieces

Steps 1–4 passed on synthetic checks. Wrap them as library functions and thread
the policy through `list_smp_encoders`, `get_model`, and `download_encoder_weights`
(nbdev: later `#| export` of the same name replaces the earlier one).


In [52]:
#| export
CURATED_ENCODER_ALIASES = {
    'convnext_tiny': 'tu-convnext_tiny',
    'efficientnetv2_s': 'tu-tf_efficientnetv2_s',
}
CURATED_TU_ENCODERS = set(CURATED_ENCODER_ALIASES.values())
_warned_experimental_tu: Set[str] = set()


def resolve_encoder_name(name: str) -> str:
    "Map a curated alias to its SMP `tu-*` name; pass other names through."
    return CURATED_ENCODER_ALIASES.get(name, name)


def _validate_encoder_name(encoder_name: str) -> None:
    "Accept SMP registry / curated `tu-*`; warn once for experimental `tu-*`; else error."
    registry = set(smp.encoders.get_encoder_names())
    if encoder_name in registry or encoder_name in CURATED_TU_ENCODERS:
        return
    if encoder_name.startswith('tu-'):
        if encoder_name not in _warned_experimental_tu:
            warnings.warn(
                f'Encoder {encoder_name!r} is an experimental tu-* (timm) encoder — '
                f'accepted, but not curated. Prefer {sorted(CURATED_ENCODER_ALIASES)} '
                f'or raw curated names {sorted(CURATED_TU_ENCODERS)}.',
                UserWarning,
                stacklevel=3,
            )
            _warned_experimental_tu.add(encoder_name)
        return
    raise ValueError(
        f'Unknown encoder {encoder_name!r}. '
        f'Use list_smp_encoders() for curated + SMP names, or a tu-* timm encoder.'
    )


In [53]:
#| export

def list_smp_encoders() -> List[str]:
    """Encoder names for discovery: SMP registry + DINOv3 + curated aliases / `tu-*`.

    Experimental (non-curated) `tu-*` encoders are accepted by `get_model` with a
    warning but are **not** listed here — use a curated name or a raw `tu-*` knowingly.
    """
    names = set(smp.encoders.get_encoder_names())
    names.update(CURATED_ENCODER_ALIASES.keys())
    names.update(CURATED_TU_ENCODERS)
    return sorted(names)



In [54]:
list_smp_encoders()

['convnext_tiny',
 'densenet121',
 'densenet161',
 'densenet169',
 'densenet201',
 'dinov3_vit_base',
 'dinov3_vit_small',
 'dpn107',
 'dpn131',
 'dpn68',
 'dpn68b',
 'dpn92',
 'dpn98',
 'efficientnet-b0',
 'efficientnet-b1',
 'efficientnet-b2',
 'efficientnet-b3',
 'efficientnet-b4',
 'efficientnet-b5',
 'efficientnet-b6',
 'efficientnet-b7',
 'efficientnetv2_s',
 'inceptionresnetv2',
 'inceptionv4',
 'mit_b0',
 'mit_b1',
 'mit_b2',
 'mit_b3',
 'mit_b4',
 'mit_b5',
 'mobilenet_v2',
 'mobileone_s0',
 'mobileone_s1',
 'mobileone_s2',
 'mobileone_s3',
 'mobileone_s4',
 'resnet101',
 'resnet152',
 'resnet18',
 'resnet34',
 'resnet50',
 'resnext101_32x16d',
 'resnext101_32x32d',
 'resnext101_32x48d',
 'resnext101_32x4d',
 'resnext101_32x8d',
 'resnext50_32x4d',
 'se_resnet101',
 'se_resnet152',
 'se_resnet50',
 'se_resnext101_32x4d',
 'se_resnext50_32x4d',
 'senet154',
 'timm-efficientnet-b0',
 'timm-efficientnet-b1',
 'timm-efficientnet-b2',
 'timm-efficientnet-b3',
 'timm-efficientnet-b4

In [57]:
#| export
def get_model(
    arch: str = 'mobilenet_v2',   # encoder name — alias kept for Hydra `model.arch`
    in_channels: int = 3,
    num_classes: int = 1,         # 1 = binary, >1 = multi-class
    pretrained: bool = True,      # ImageNet weights (ignored when weights_path is set)
    weights_path: Optional[str] = None,  # local encoder .pth from download_encoder_weights
    device: str = 'cpu',
    architecture: str = 'unet',   # smp head: unet | fpn | deeplabv3plus | ...
    encoder: Optional[str] = None,  # explicit encoder name; overrides `arch` when set
) -> nn.Module:
    """Build an smp model returning raw logits `(B, num_classes, H, W)`.

    Pass `weights_path` on air-gapped HPC instead of downloading ImageNet weights.
    Use `list_smp_architectures()` and `list_smp_encoders()` to discover valid names.
    Curated aliases: `convnext_tiny` → `tu-convnext_tiny`, `efficientnetv2_s` →
    `tu-tf_efficientnetv2_s`. Other `tu-*` names warn once then build. Unknown
    non-`tu-*` names raise. `dinov3_vit_small` / `dinov3_vit_base` use the DPT
    reassemble path below.

    Example:
        >>> m = get_model('convnext_tiny', in_channels=1, num_classes=1, pretrained=False)
        >>> tuple(m(torch.randn(1, 1, 64, 64)).shape)
        (1, 1, 64, 64)
    """
    encoder_name = resolve_encoder_name(encoder if encoder is not None else arch)
    arch_key = _normalize_architecture(architecture)
    if arch_key not in SMP_ARCHITECTURES:
        raise ValueError(
            f"Unknown architecture {architecture!r}. "
            f"Choose one of: {', '.join(list_smp_architectures())}"
        )
    _validate_encoder_name(encoder_name)
    use_imagenet = pretrained and weights_path is None
    model_cls = SMP_ARCHITECTURES[arch_key]
    if encoder_name.startswith('dinov3_'):
        smp.encoders.encoders[encoder_name]['params']['pretrained'] = use_imagenet
        encoder_weights = None
    else:
        encoder_weights = 'imagenet' if use_imagenet else None
    model = model_cls(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )
    if weights_path is not None:
        _load_local_encoder_weights(model, weights_path, device)
    return model



In [58]:
#| eval: false
# downloads DINOv3 weights from the timm hub — needs network + ~2 GB cache; run manually
mdl = get_model('dinov3_vit_small', in_channels=1, num_classes=1, pretrained=True)


model.safetensors: reconstructing file:   0%|          |  0.00B / 86.4MB            

model.safetensors: downloading bytes:           |  0.00B            

In [60]:
#| export
def download_encoder_weights(
    encoder_name: str,   # smp encoder name or curated alias
    save_path: str,      # local .pth path to write
) -> None:
    """Download ImageNet encoder weights to a portable local file.

    Run **once** on any machine that has internet access, copy the file to your
    air-gapped HPC, then use `get_model(weights_path=...)`. Curated aliases
    (`convnext_tiny`, …) are resolved to their `tu-*` names first.

    Example:
        >>> import tempfile, os
        >>> p = os.path.join(tempfile.mkdtemp(), 'enc.pth')
        >>> download_encoder_weights('mobilenet_v2', p)   # doctest: +SKIP
        >>> os.path.exists(p)                              # doctest: +SKIP
        True
    """
    from pathlib import Path
    encoder_name = resolve_encoder_name(encoder_name)
    _validate_encoder_name(encoder_name)
    tmp = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                   in_channels=3, classes=1, activation=None)
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(tmp.encoder.state_dict(), save_path)
    n = len(tmp.encoder.state_dict())
    size_mb = Path(save_path).stat().st_size / 1e6
    print(f'Saved {encoder_name} encoder  ({n} tensors, {size_mb:.1f} MB)  →  {save_path}')
    del tmp


In [61]:
# test: curated get_model / list / policy (same checks as steps 1–4, via exports)
import warnings

assert resolve_encoder_name('convnext_tiny') == 'tu-convnext_tiny'
assert resolve_encoder_name('mobilenet_v2') == 'mobilenet_v2'

for name in ('convnext_tiny', 'tu-convnext_tiny', 'efficientnetv2_s', 'tu-tf_efficientnetv2_s'):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        m = get_model(name, architecture='unet', in_channels=1, num_classes=1, pretrained=False)
        assert tuple(m(torch.randn(1, 1, 64, 64)).shape) == (1, 1, 64, 64), name
        tu_warns = [x for x in w if issubclass(x.category, UserWarning) and 'experimental' in str(x.message).lower()]
        assert not tu_warns, f'unexpected warning for curated {name}: {tu_warns}'

encs = set(list_smp_encoders())
assert {'convnext_tiny', 'efficientnetv2_s', 'tu-convnext_tiny', 'tu-tf_efficientnetv2_s'} <= encs
assert {'mobilenet_v2', 'dinov3_vit_small'} <= encs

experimental = 'tu-mobilenetv3_small_100'
with warnings.catch_warnings(record=True) as w1:
    warnings.simplefilter('always')
    m_exp = get_model(experimental, architecture='unet', in_channels=1, num_classes=1, pretrained=False)
    assert tuple(m_exp(torch.randn(1, 1, 64, 64)).shape) == (1, 1, 64, 64)
    assert any('experimental' in str(x.message).lower() and experimental in str(x.message) for x in w1)
with warnings.catch_warnings(record=True) as w2:
    warnings.simplefilter('always')
    get_model(experimental, architecture='unet', in_channels=1, num_classes=1, pretrained=False)
    assert not [x for x in w2 if experimental in str(x.message)]

try:
    get_model('not_a_real_encoder_xyz', pretrained=False)
    assert False, 'expected ValueError'
except ValueError as e:
    assert 'not_a_real_encoder_xyz' in str(e)


### Hydra packages and ONNX smoke

Config packages live under `configs/model/` (YAML only). Compose checks belong
next to the factory; ONNX smoke uses the curated `get_model` path for CNN aliases
(`dinov3_*` ONNX is heavier — covered elsewhere).


In [62]:
# experiment: Hydra model packages compose (not exported — YAML only)
from pathlib import Path
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

here = Path.cwd()
cfg_dir = next(
    (str(p.resolve()) for p in (here / 'configs', here.parent / 'configs') if p.is_dir()),
    None,
)
assert cfg_dir is not None, f'configs/ not found from cwd={here}'
GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=cfg_dir, version_base='1.3'):
    lib = compose(config_name='train')
    ml = compose(config_name='train_multilabel_3head')
    print('library default arch:', lib.model.arch)
    print('multilabel recipe arch:', ml.model.arch)
    print('shut-off from multilabel:', compose(config_name='train_multilabel_3head', overrides=['model=mobilenet_v2']).model.arch)


library default arch: mobilenet_v2
multilabel recipe arch: dinov3_vit_small
shut-off from multilabel: mobilenet_v2


In [63]:
# verify: library MobileNet; multilabel DINOv3-S; shut-off MobileNet; aliases compose
from pathlib import Path
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

here = Path.cwd()
cfg_dir = next(
    (str(p.resolve()) for p in (here / 'configs', here.parent / 'configs') if p.is_dir()),
    None,
)
GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=cfg_dir, version_base='1.3'):
    assert compose(config_name='train').model.arch == 'mobilenet_v2'
    assert compose(config_name='train_multilabel_3head').model.arch == 'dinov3_vit_small'
    assert compose(config_name='train_multilabel_3head', overrides=['model=mobilenet_v2']).model.arch == 'mobilenet_v2'
    assert compose(config_name='train', overrides=['model=convnext_tiny']).model.arch == 'convnext_tiny'
    assert compose(config_name='train', overrides=['model=efficientnetv2_s']).model.arch == 'efficientnetv2_s'
print('hydra compose ok')


hydra compose ok


In [64]:
# test: HYDRA_ENCODER_PRESET_COMPOSE
from pathlib import Path
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

here = Path.cwd()
cfg_dir = next(
    (str(p.resolve()) for p in (here / 'configs', here.parent / 'configs') if p.is_dir()),
    None,
)
assert cfg_dir is not None, f'configs/ not found from cwd={here}'

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=cfg_dir, version_base='1.3'):
    cfg = compose(config_name='train')
    assert cfg.model.arch == 'mobilenet_v2'
    assert cfg.model.architecture == 'unet'
    cfg_dino = compose(config_name='train', overrides=['model=dinov3_vit_small'])
    assert cfg_dino.model.arch == 'dinov3_vit_small'
    cfg_cn = compose(config_name='train', overrides=['model=convnext_tiny'])
    assert cfg_cn.model.arch == 'convnext_tiny'
    cfg_eff = compose(config_name='train', overrides=['model=efficientnetv2_s'])
    assert cfg_eff.model.arch == 'efficientnetv2_s'
    cfg_ml = compose(config_name='train_multilabel_3head')
    assert cfg_ml.model.arch == 'dinov3_vit_small'
    cfg_off = compose(config_name='train_multilabel_3head', overrides=['model=mobilenet_v2'])
    assert cfg_off.model.arch == 'mobilenet_v2'


In [65]:
# test: CURATED_ENCODER_ONNX_SMOKE
# FP32 ONNX smoke for curated CNN builds via get_model (fixed spatial size).
# dinov3_* ONNX is covered separately (pretrained=False still heavy); skip here.
import tempfile
from pathlib import Path
from segmentation_tools_pt.export import export_onnx

for name in ('mobilenet_v2', 'convnext_tiny', 'efficientnetv2_s'):
    m = get_model(name, architecture='unet', in_channels=1, num_classes=3, pretrained=False)
    with tempfile.TemporaryDirectory() as d:
        out = Path(d) / f'{name}.onnx'
        export_onnx(m, str(out), in_channels=1, input_size=64, opset=18, verify=True)
        assert out.exists() and out.stat().st_size > 0, name


In [66]:
# test
groups = get_param_groups(m1)
assert len(groups) == 2 and groups[0]['lr'] < groups[1]['lr']
freeze_encoder(m1)
assert all(not p.requires_grad for p in m1.encoder.parameters())
unfreeze_encoder(m1)
assert all(p.requires_grad for p in m1.encoder.parameters())

# fallback: a plain (non-SMP) model has no .encoder -- one param group, freeze/unfreeze are no-ops
toy = nn.Conv2d(3, 1, 3, padding=1)
toy_groups = get_param_groups(toy)
assert len(toy_groups) == 1 and toy_groups[0]['lr'] == 1e-3
assert set(toy_groups[0]['params']) == set(toy.parameters())
freeze_encoder(toy)   # no .encoder -> no-op, must not raise
unfreeze_encoder(toy)
assert all(p.requires_grad for p in toy.parameters())

## Custom UNet (`model.source: custom_unet`)

Problem: `get_model` only builds SMP architectures. Industrial crops are often small, grayscale and
unlike ImageNet, so we also want a library-owned classic UNet that trains from scratch and is
configurable from Hydra YAML. Design: `docs/superpowers/specs/2026-10-01-custom-unet-smp-factory-design.md`
(tickets #61 tracer bullet, #62 depth and norm).

Scope so far: double-conv blocks, GroupNorm, explicit integer `encoder_depth` (#61), plus `recommend`/`auto`
depth chosen from the crop size and a selectable `norm: group | batch | instance` (#62, Step 4).
`block_style: residual` (#63, Step 5), `bottleneck: aspp` (#64, Step 6) and `bottleneck: attention` (#65, Step 7).

Plan, in order:

1. a norm helper that always picks a group count dividing the channel count, and a double-conv block;
2. an encoder returning one feature map per stage, a decoder, and a 1x1 head, wrapped as `CustomUNet`
   with SMP-compatible `.encoder` / `.decoder` / `.segmentation_head` so `get_param_groups` and `freeze_encoder` work;
3. `get_model_from_cfg(cfg, in_channels, crop_size=...)` branching on `model.source` (`smp` default);
4. `resolve_encoder_depth` (`recommend` / `auto` / int) and the norm choice, each tried as an experiment first;
5. a residual block selected by `block_style`, through one block table;
6. an ASPP bottleneck and 7. a self-attention bottleneck, both through one bottleneck table.

### Step 1 - norm helper and double-conv block

In [ ]:
# experiment: GroupNorm needs num_groups | channels; pick the largest divisor <= requested
def _group_count(channels, want=8):
    return max(g for g in range(1, min(want, channels) + 1) if channels % g == 0)

_blk = nn.Sequential(
    nn.Conv2d(1, 12, 3, padding=1, bias=False), nn.GroupNorm(_group_count(12), 12), nn.ReLU(inplace=True),
    nn.Conv2d(12, 12, 3, padding=1, bias=False), nn.GroupNorm(_group_count(12), 12), nn.ReLU(inplace=True))
print(_group_count(12), _group_count(32), _group_count(5), _blk(torch.randn(2, 1, 16, 16)).shape)

In [ ]:
# verify: 12 channels cannot use 8 groups -> 6; prime channel counts fall back to 1; spatial size preserved
assert _group_count(12) == 6 and _group_count(32) == 8 and _group_count(5) == 5 and _group_count(7, 4) == 1
assert tuple(_blk(torch.randn(2, 1, 16, 16)).shape) == (2, 12, 16, 16)

### Step 2 - encoder, decoder, head

Stage `i` has `min(base_channels * 2**i, max_channels)` channels; stages `1..depth` start with a 2x2 max-pool,
so depth 4 on a 64x64 crop ends at a 4x4 bottleneck. The decoder mirrors the encoder (transposed conv, concat the skip, double conv).

In [ ]:
# experiment: channel plan and spatial plan for depth 4, base 32, max 512
base, mx, depth = 32, 512, 4
chs = [min(base * 2**i, mx) for i in range(depth + 1)]
sizes = [64 // 2**i for i in range(depth + 1)]
print(chs, sizes)

In [ ]:
# verify: channel plan is capped and the bottleneck is 64 / 2**depth
assert chs == [32, 64, 128, 256, 512] and sizes[-1] == 4
assert [min(32 * 2**i, 64) for i in range(4)] == [32, 64, 64, 64]

#### Run the pieces before promoting them

The plan above is only arithmetic. Here we actually run a tiny encoder and decoder built from plain `nn` layers
(throwaway versions; the exported classes below are the same idea) and look at the shape of every stage.

In [ ]:
# experiment: encoder = one feature map per stage; print the shape at every stage
_chs = [8, 16, 32, 64]                       # depth 3, base 8
def _dc(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU())
_enc = nn.ModuleList([_dc(1, _chs[0])] + [nn.Sequential(nn.MaxPool2d(2), _dc(_chs[k-1], _chs[k])) for k in range(1, 4)])
_x, _feats = torch.randn(1, 1, 64, 64), []
for _s in _enc:
    _x = _s(_x); _feats.append(_x)
print([tuple(f.shape) for f in _feats])

In [ ]:
# verify: channels follow the plan, spatial size halves per stage, bottleneck is 64 / 2**3
assert [f.shape[1] for f in _feats] == _chs
assert [f.shape[-1] for f in _feats] == [64, 32, 16, 8]

In [ ]:
# experiment: decoder = upsample deepest map, concat the skip, conv; print shape after every step
_ups = nn.ModuleList([nn.ConvTranspose2d(_chs[k+1], _chs[k], 2, stride=2) for k in reversed(range(3))])
_dec = nn.ModuleList([_dc(2 * _chs[k], _chs[k]) for k in reversed(range(3))])
_y, _trace = _feats[-1], []
for _u, _b, _skip in zip(_ups, _dec, reversed(_feats[:-1])):
    _y = _b(torch.cat([_skip, _u(_y)], dim=1)); _trace.append(tuple(_y.shape))
print(_trace)

In [ ]:
# verify: decoder walks back to the full-resolution, base-channel map; a 1x1 head gives num_classes logits
assert _trace == [(1, 32, 16, 16), (1, 16, 32, 32), (1, 8, 64, 64)]
assert tuple(nn.Conv2d(_chs[0], 3, 1)(_y).shape) == (1, 3, 64, 64)
# odd input: pooling floors (37 -> 18), the upsampled map (36) is one pixel short of the skip (37) -> needs a resize
_o = nn.ConvTranspose2d(8, 8, 2, stride=2)(nn.MaxPool2d(2)(torch.randn(1, 8, 37, 37)))
assert _o.shape[-1] == 36 != 37

### Step 5 - residual block (#63)

`block_style: residual` swaps the double conv for a residual block: `Conv3x3 -> Norm -> ReLU -> Conv3x3 -> Norm`,
then add the input (the skip) and apply ReLU. When the channel count changes, the skip is a bias-free 1x1 conv so
the shapes match; otherwise it is the identity. There is no third plain conv in v1. In the decoder the block sees the
concatenated skip + upsampled map (`2 * ch` in, `ch` out), so it needs the 1x1 projection too.

One table maps the `block_style` name to the block class (like `_NORM_FACTORIES` does for norms), and the name is
checked before any model is built. First try the idea inline on a tensor.

In [ ]:
# experiment: residual = relu(norm(conv(relu(norm(conv(x))))) + skip(x)); projection only when channels change
def _res(i, o):
    c1, n1 = nn.Conv2d(i, o, 3, padding=1, bias=False), nn.GroupNorm(_group_count(o), o)
    c2, n2 = nn.Conv2d(o, o, 3, padding=1, bias=False), nn.GroupNorm(_group_count(o), o)
    sk = nn.Identity() if i == o else nn.Conv2d(i, o, 1, bias=False)
    return lambda t: torch.relu(n2(c2(torch.relu(n1(c1(t))))) + sk(t)), sk
_f, _sk = _res(4, 8); _g, _sk2 = _res(8, 8)
print(tuple(_f(torch.randn(2, 4, 16, 16)).shape), tuple(_g(torch.randn(2, 8, 16, 16)).shape), type(_sk).__name__, type(_sk2).__name__)

In [ ]:
# verify: shape follows out_ch, 1x1 skip only when channels differ, and the 2*ch decoder input needs it
assert tuple(_f(torch.randn(2, 4, 16, 16)).shape) == (2, 8, 16, 16) and tuple(_g(torch.randn(2, 8, 16, 16)).shape) == (2, 8, 16, 16)
assert isinstance(_sk, nn.Conv2d) and isinstance(_sk2, nn.Identity)
assert isinstance(_res(16, 8)[1], nn.Conv2d)                                             # decoder block: 2*ch -> ch

### Step 6 - ASPP bottleneck (#64)

`bottleneck: aspp` puts an ASPP block (Atrous Spatial Pyramid Pooling) on the smallest feature map, between encoder and
decoder. "Atrous" means the 3x3 conv skips pixels (`dilation=rate`), so a high rate sees far across the map for the
price of a 3x3. The block runs these in parallel on the bottleneck map `x` with `C` channels:

- one 3x3 conv per rate in `aspp_rates` (default `[1, 6, 12, 18]`, rate 1 is a plain 3x3), each `C // 4` channels, Norm, ReLU
- an image-pooling branch: global average -> 1x1 conv -> ReLU, broadcast back to the map size (no norm here: a norm over
  one pixel per channel is undefined for InstanceNorm and needs batch > 1 for BatchNorm)

then concat all branches and project with a 1x1 conv back to `C`, so the module keeps `(B, C, H, W)` and the UNet around it
does not change. With `bottleneck: none` the bottleneck is `nn.Identity()`, so the default network is untouched.

A rate `r` only helps while the map is bigger than `r` (padding `r` on a map of side `<= r` leaves only the centre tap).
Below 16 px the factory therefore drops rates that are not smaller than the bottleneck side, and falls back to `[1]` if
none is left. First try both ideas inline.

In [ ]:
# experiment: parallel dilated convs + pooled branch -> concat -> 1x1 project, shape kept; then the trim rule
C, rates = 32, [1, 6, 12, 18]
_br = [nn.Conv2d(C, C // 4, 3, padding=r, dilation=r, bias=False) for r in rates]
_pool, _proj = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(C, C // 4, 1), nn.ReLU()), nn.Conv2d(5 * (C // 4), C, 1, bias=False)
def _aspp_inline(t):
    p = nn.functional.interpolate(_pool(t), size=t.shape[-2:], mode='nearest')
    return _proj(torch.cat([b(t) for b in _br] + [p], dim=1))
for hw in [(16, 16), (8, 8), (5, 7)]:
    print(hw, tuple(_aspp_inline(torch.randn(2, C, *hw)).shape))
def _trim_inline(rs, side): return list(rs) if side >= 16 else ([r for r in rs if r < side] or [1])
print({side: _trim_inline(rates, side) for side in (32, 16, 15, 8, 6, 1)})

In [ ]:
# verify: size and channel count survive for odd and tiny maps; trim keeps all from 16 px, drops rates >= side below, never empties
for hw in [(16, 16), (8, 8), (5, 7), (1, 1)]:
    assert tuple(_aspp_inline(torch.randn(2, C, *hw)).shape) == (2, C, *hw), hw
assert _trim_inline(rates, 32) == _trim_inline(rates, 16) == rates
assert _trim_inline(rates, 15) == [1, 6, 12] and _trim_inline(rates, 8) == [1, 6] and _trim_inline(rates, 6) == [1] and _trim_inline(rates, 1) == [1]
assert _trim_inline([6, 12], 4) == [1]                                                  # nothing fits -> plain 3x3

### Step 7 - attention bottleneck (#65)

`bottleneck: attention` puts one self-attention layer on the smallest feature map. A conv only sees a small neighbourhood;
attention lets every position of the bottleneck look at every other one, which gives global context cheaply because the
bottleneck map is small. The plan, on a bottleneck map `x` with `C` channels:

- flatten `(B, C, H, W)` into `H*W` tokens `(B, H*W, C)`
- LayerNorm the tokens, project to query / key / value (one linear layer), split into `heads` heads
- scaled dot-product attention, merge the heads, project back to `C`
- add the result to the tokens (a residual add, so the layer starts close to an identity and cannot hurt much), then fold back to `(B, C, H, W)`

`attention_heads` (default 4) must divide `C`, because each head gets `C // heads` channels. `bottleneck` is one value, so
`aspp` and `attention` are mutually exclusive; `aspp_rates` is ignored for attention and `attention_heads` for the others.

Two cost notes. Memory: attention compares every token with every other, so it grows with `(H*W)^2`. A 512 crop at
`encoder_depth: 4` gives a `32x32` map = 1024 tokens (fine); depth 3 gives `64x64` = 4096 tokens (heavy).
ONNX: `nn.MultiheadAttention` bakes the token count into a reshape when exported, so a graph exported with dynamic H/W
only works at the traced size (reproduced in the tests below: onnxruntime fails with a Reshape error at any other size). Writing the attention
out with `scaled_dot_product_attention` keeps H/W dynamic (checked in the tests below), so that is what we use. Both results are
from the legacy exporter (`dynamo=False`, opset 18), the one verified; the `export_onnx` seam is covered too.

One more property to know: there is no positional encoding. The layer is permutation-equivariant on its own, and the
layout information comes from the neighbouring conv features.
First try the idea inline.

In [ ]:
# experiment: tokens -> LayerNorm -> qkv -> SDPA -> project -> residual -> back to a map; and does a pixel reach the others?
C, heads = 32, 4
_ln, _qkv, _out = nn.LayerNorm(C), nn.Linear(C, 3 * C), nn.Linear(C, C)
def _attn_inline(x):
    t = x.flatten(2).transpose(1, 2)
    q, k, v = [z.unflatten(-1, (heads, -1)).transpose(1, 2) for z in _qkv(_ln(t)).chunk(3, dim=-1)]
    a = nn.functional.scaled_dot_product_attention(q, k, v).transpose(1, 2).flatten(2)
    return (t + _out(a)).transpose(1, 2).reshape(x.shape)
for hw in [(8, 8), (5, 7), (1, 1)]:
    print(hw, tuple(_attn_inline(torch.randn(2, C, *hw)).shape))
_x = torch.randn(1, C, 4, 4); _x2 = _x.clone(); _x2[..., 0, 0] += 1
print('far pixel changed:', bool((_attn_inline(_x) - _attn_inline(_x2))[..., 3, 3].abs().max() > 0))

In [ ]:
# verify: size and channels survive odd and 1x1 maps; a change at one position reaches a far position (attention mixes space)
for hw in [(8, 8), (5, 7), (1, 1)]:
    assert tuple(_attn_inline(torch.randn(2, C, *hw)).shape) == (2, C, *hw), hw
assert (_attn_inline(_x) - _attn_inline(_x2))[..., 3, 3].abs().max() > 0

### Promote - export the working pieces

In [ ]:
#| export
def _group_count(channels: int, want: int = 8) -> int:
    "Largest group count <= `want` that divides `channels` (GroupNorm needs an exact divisor)."
    return max(g for g in range(1, min(want, channels) + 1) if channels % g == 0)

# the one place that knows which norms exist: name -> factory(channels, num_groups); add a norm by adding a row
_NORM_FACTORIES = {
    'group': lambda channels, num_groups: nn.GroupNorm(_group_count(channels, num_groups), channels),
    'batch': lambda channels, num_groups: nn.BatchNorm2d(channels),
    'instance': lambda channels, num_groups: nn.InstanceNorm2d(channels, affine=True),
}
_NORMS = tuple(_NORM_FACTORIES)

def _check_choice(value, table, label: str) -> None:
    "Raise a `ValueError` naming the config key `label` unless `value` is a key of `table`."
    try:
        known = value in table
    except TypeError:                  # unhashable (a list or dict from YAML) cannot be a key
        known = False
    if not known:
        raise ValueError(f"{label}={value!r} is not supported; choose one of {list(table)}")

def _positive_int(value, label: str) -> int:
    "Return `value` if it is a real int >= 1 (bool is not accepted); otherwise raise a `ValueError` naming `label`."
    if isinstance(value, bool) or not isinstance(value, int) or value < 1:
        raise ValueError(f'{label} must be a positive integer, got {value!r}')
    return value

def _check_norm(norm, label: str = 'norm') -> None:
    _check_choice(norm, _NORM_FACTORIES, label)

def _norm_layer(norm: str, channels: int, num_groups: int) -> nn.Module:
    _check_norm(norm)
    return _NORM_FACTORIES[norm](channels, num_groups)

class _DoubleConv(nn.Sequential):
    "Conv3x3 -> Norm -> ReLU, twice (Ronneberger block)."
    def __init__(self, in_ch: int, out_ch: int, norm: str = 'group', num_groups: int = 8):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), _norm_layer(norm, out_ch, num_groups), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), _norm_layer(norm, out_ch, num_groups), nn.ReLU(inplace=True),
        )

class _ResidualBlock(nn.Module):
    "Conv3x3 -> Norm -> ReLU -> Conv3x3 -> Norm, add the skip (1x1 conv when channels differ, else identity), ReLU."
    def __init__(self, in_ch: int, out_ch: int, norm: str = 'group', num_groups: int = 8):
        super().__init__()
        self.conv1, self.norm1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), _norm_layer(norm, out_ch, num_groups)
        self.conv2, self.norm2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), _norm_layer(norm, out_ch, num_groups)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        y = self.norm2(self.conv2(self.act(self.norm1(self.conv1(x)))))
        return self.act(y + self.skip(x))

# the one place that knows the block styles, so adding one is a one-row change and the validator and the builders can't disagree
_BLOCK_CLASSES = {'double': _DoubleConv, 'residual': _ResidualBlock}
_DEFAULT_BLOCK_STYLE = 'double'
_BLOCK_STYLES = tuple(_BLOCK_CLASSES)

def _check_block_style(block_style, label: str = 'block_style') -> None:
    _check_choice(block_style, _BLOCK_CLASSES, label)

_DEFAULT_ASPP_RATES = (1, 6, 12, 18)
_DEFAULT_BASE_CHANNELS, _DEFAULT_MAX_CHANNELS = 32, 512  # channels of the first stage; cap for the doubling
_ASPP_FULL_SIZE = 16  # design spec: below 16 px per bottleneck side the large rates run past the map, so they are trimmed

def trim_aspp_rates(
    rates: Sequence[int],  # dilation rates asked for in `model.custom_unet.aspp_rates`
    size: int              # bottleneck side in pixels (crop_size // 2**encoder_depth)
) -> List[int]:
    """Rates that still see real pixels on a `size`-pixel bottleneck.

    From 16 px up all rates are kept; below, a rate must be smaller than the side (padding `r` on a map of
    side `<= r` leaves only the centre tap). Falls back to `[1]`, a plain 3x3, when nothing fits.

    Example:
        >>> trim_aspp_rates([1, 6, 12, 18], 8)
        [1, 6]
    """
    if size >= _ASPP_FULL_SIZE:
        return list(rates)
    return [r for r in rates if r < size] or [1]

class _ASPP(nn.Module):
    "Parallel dilated 3x3 convs (one per rate) + image pooling, concat, 1x1 project back to `channels`; keeps `(B, C, H, W)`."
    def __init__(self, channels: int, rates: Sequence[int], norm: str = 'group', num_groups: int = 8):
        super().__init__()
        branch = max(channels // 4, 1)
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(channels, branch, 3, padding=r, dilation=r, bias=False), _norm_layer(norm, branch, num_groups), nn.ReLU(inplace=True))
            for r in rates])
        # no norm on the pooled branch: it is one pixel per channel, which InstanceNorm rejects and BatchNorm needs batch > 1 for
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, branch, 1), nn.ReLU(inplace=True))
        self.project = nn.Sequential(nn.Conv2d((len(self.branches) + 1) * branch, channels, 1, bias=False),
                                     _norm_layer(norm, channels, num_groups), nn.ReLU(inplace=True))
    def forward(self, x):
        pooled = nn.functional.interpolate(self.pool(x), size=x.shape[-2:], mode='nearest')
        return self.project(torch.cat([b(x) for b in self.branches] + [pooled], dim=1))

_DEFAULT_ATTENTION_HEADS = 4

# scaled_dot_product_attention instead of nn.MultiheadAttention: the latter bakes the token count into an ONNX reshape, so a graph
# exported with dynamic H/W would only run at the traced size. Memory is O(tokens^2): 1024 tokens at a 32x32 map, 4096 at 64x64.
class _AttentionBottleneck(nn.Module):
    "One pre-norm self-attention layer over the `H*W` positions of the map, with a residual add; keeps `(B, C, H, W)`."
    def __init__(self, channels: int, heads: int = _DEFAULT_ATTENTION_HEADS):
        super().__init__()
        _check_attention_heads(heads, channels)
        self.heads = heads
        self.norm = nn.LayerNorm(channels)
        self.qkv = nn.Linear(channels, 3 * channels)
        self.proj = nn.Linear(channels, channels)
    def forward(self, x):
        tokens = x.flatten(2).transpose(1, 2)
        q, k, v = [z.unflatten(-1, (self.heads, -1)).transpose(1, 2) for z in self.qkv(self.norm(tokens)).chunk(3, dim=-1)]
        mixed = nn.functional.scaled_dot_product_attention(q, k, v).transpose(1, 2).flatten(2)
        return (tokens + self.proj(mixed)).transpose(1, 2).reshape(x.shape)

def _prepare_none(knobs, channels, size, prefix): return knobs

def _prepare_aspp(knobs, channels, size, prefix):
    rates = _aspp_rates(knobs['aspp_rates'], f'{prefix}aspp_rates')
    return {**knobs, 'aspp_rates': rates if size is None else trim_aspp_rates(rates, _positive_int(size, 'bottleneck_size'))}

def _prepare_attention(knobs, channels, size, prefix):
    _check_attention_heads(knobs['attention_heads'], channels, f'{prefix}attention_heads')
    return knobs

# the one place that knows the bottlenecks: name -> (build(channels, knobs, norm, num_groups), prepare(knobs, channels, size, prefix)).
# `prepare` checks (and for aspp trims) only the knobs its bottleneck reads; the others are ignored, so adding one is one row
_BOTTLENECKS = {
    'none': (lambda channels, knobs, norm, num_groups: nn.Identity(), _prepare_none),
    'aspp': (lambda channels, knobs, norm, num_groups: _ASPP(channels, knobs['aspp_rates'], norm, num_groups), _prepare_aspp),
    'attention': (lambda channels, knobs, norm, num_groups: _AttentionBottleneck(channels, knobs['attention_heads']), _prepare_attention),
}

def _check_bottleneck(bottleneck, label: str = 'bottleneck') -> None:
    "Raise a `ValueError` naming `label` unless `bottleneck` is a row of `_BOTTLENECKS`."
    _check_choice(bottleneck, _BOTTLENECKS, label)

def _check_attention_heads(heads, channels: int, label: str = 'attention_heads') -> int:
    "`heads` if it is a positive int that divides `channels` (each head gets `channels // heads`), else a `ValueError` naming `label`."
    _positive_int(heads, label)
    if channels % heads:
        raise ValueError(f'{label}={heads} must divide the bottleneck channel count {channels}')
    return heads

def _aspp_rates(rates, label: str = 'aspp_rates') -> List[int]:
    "`rates` as a list of positive ints, or a `ValueError` naming `label`."
    if isinstance(rates, str) or not isinstance(rates, Sequence) or len(rates) == 0:
        raise ValueError(f'{label} must be a non-empty list of positive integers, got {rates!r}')
    return [_positive_int(r, label) for r in rates]

class _CustomUNetEncoder(nn.Module):
    "`depth + 1` stages; `forward` returns the feature map of every stage (shallow to deep)."
    def __init__(self, in_channels, channels, block):
        super().__init__()
        stages = [block(in_channels, channels[0])]
        for i in range(1, len(channels)):
            stages.append(nn.Sequential(nn.MaxPool2d(2), block(channels[i - 1], channels[i])))
        self.stages = nn.ModuleList(stages)
    def forward(self, x):
        feats = []
        for stage in self.stages:
            x = stage(x)
            feats.append(x)
        return feats

class _CustomUNetDecoder(nn.Module):
    "Mirror of the encoder: upsample, concat the skip, conv block. Returns the full-resolution map."
    def __init__(self, channels, block):
        super().__init__()
        num_ups = len(channels) - 1
        self.ups = nn.ModuleList([nn.ConvTranspose2d(channels[i + 1], channels[i], 2, stride=2) for i in reversed(range(num_ups))])
        self.blocks = nn.ModuleList([block(2 * channels[i], channels[i]) for i in reversed(range(num_ups))])
    def forward(self, feats):
        x = feats[-1]
        for up, block, skip in zip(self.ups, self.blocks, reversed(feats[:-1])):
            x = up(x)
            # odd sizes lose a pixel when pooling; always resizing (an identity for even sizes) keeps this
            # traceable, so an ONNX graph exported with dynamic H/W also handles odd inputs
            x = nn.functional.interpolate(x, size=skip.shape[-2:], mode='nearest')
            x = block(torch.cat([skip, x], dim=1))
        return x

def _channel_plan(base_channels: int, max_channels: int, depth: int) -> List[int]:
    "Channel count of each of the `depth + 1` stages: doubling from `base_channels`, capped at `max_channels`."
    return [min(base_channels * 2**i, max_channels) for i in range(depth + 1)]

class CustomUNet(nn.Module):
    """Library-owned classic UNet returning raw logits `(B, num_classes, H, W)`.

    Exposes `.encoder` / `.decoder` / `.segmentation_head` like an smp model, so
    `get_param_groups` and `freeze_encoder` work unchanged.

    `block_style` picks the conv block (`double` or `residual`, see `_BLOCK_CLASSES`).
    `bottleneck` (`none`, `aspp` or `attention`) adds a size-preserving `.bottleneck` module between encoder and decoder.
    `attention_heads` must divide the bottleneck channels and is used only by `attention`.
    `aspp_rates` are used as given, unless `bottleneck_size` (the bottleneck side in pixels, `crop_size // 2**encoder_depth`)
    is passed: then `trim_aspp_rates` drops the rates the map is too small for.

    Example:
        >>> m = CustomUNet(in_channels=1, num_classes=1, encoder_depth=4)
        >>> tuple(m(torch.randn(1, 1, 64, 64)).shape)
        (1, 1, 64, 64)
    """
    def __init__(self, in_channels: int = 1, num_classes: int = 1, encoder_depth: int = 4,
                 base_channels: int = _DEFAULT_BASE_CHANNELS, max_channels: int = _DEFAULT_MAX_CHANNELS, norm: str = 'group', num_groups: int = 8,
                 block_style: str = _DEFAULT_BLOCK_STYLE, bottleneck: str = 'none', aspp_rates: Sequence[int] = _DEFAULT_ASPP_RATES,
                 bottleneck_size: Optional[int] = None, attention_heads: int = _DEFAULT_ATTENTION_HEADS):
        super().__init__()
        if encoder_depth < 1:
            raise ValueError(f'encoder_depth must be >= 1, got {encoder_depth}')
        _check_norm(norm)
        _check_block_style(block_style)
        _check_bottleneck(bottleneck)
        channels = _channel_plan(base_channels, max_channels, encoder_depth)
        build, prepare = _BOTTLENECKS[bottleneck]
        knobs = prepare({'aspp_rates': aspp_rates, 'attention_heads': attention_heads}, channels[-1], bottleneck_size, '')
        block = partial(_BLOCK_CLASSES[block_style], norm=norm, num_groups=num_groups)
        self.encoder = _CustomUNetEncoder(in_channels, channels, block)
        self.bottleneck = build(channels[-1], knobs, norm, num_groups)
        self.decoder = _CustomUNetDecoder(channels, block)
        self.segmentation_head = nn.Conv2d(channels[0], num_classes, 1)
    def decoder_side_parameters(self):
        "Parameters trained at the decoder learning rate: decoder, head and the bottleneck (random-init, like the decoder)."
        return [*self.decoder.parameters(), *self.segmentation_head.parameters(), *self.bottleneck.parameters()]
    def forward(self, x):
        feats = self.encoder(x)
        feats[-1] = self.bottleneck(feats[-1])
        return self.segmentation_head(self.decoder(feats))

In [ ]:
# test: depth 4, 1x1x64x64 in -> (1, num_classes, 64, 64) out; depth and odd sizes behave
for k in (1, 3):
    assert tuple(CustomUNet(1, k, 4)(torch.randn(1, 1, 64, 64)).shape) == (1, k, 64, 64)
m_d3 = CustomUNet(2, 1, encoder_depth=3, base_channels=12)  # 12 channels -> 6 groups
assert len(m_d3.encoder.stages) == 4 and tuple(m_d3(torch.randn(2, 2, 40, 40)).shape) == (2, 1, 40, 40)
assert tuple(CustomUNet(1, 1, 3, base_channels=8)(torch.randn(1, 1, 37, 41)).shape) == (1, 1, 37, 41)
try: CustomUNet(encoder_depth=0); raise AssertionError('depth 0 accepted')
except ValueError: pass
try: CustomUNet(norm='layer'); raise AssertionError('unknown norm accepted')
except ValueError: pass

In [ ]:
# test: SMP-compatible grouping -> get_param_groups / freeze_encoder / unfreeze_encoder work on the custom model
cm = CustomUNet(1, 1, 3, base_channels=8)
g = get_param_groups(cm, encoder_lr=1e-4, decoder_lr=1e-3)
assert len(g) == 2 and g[0]['lr'] < g[1]['lr']
assert {id(p) for grp in g for p in grp['params']} == {id(p) for p in cm.parameters()}
freeze_encoder(cm)
assert all(not p.requires_grad for p in cm.encoder.parameters())
assert all(p.requires_grad for p in cm.decoder.parameters()) and all(p.requires_grad for p in cm.segmentation_head.parameters())
unfreeze_encoder(cm)
assert all(p.requires_grad for p in cm.parameters())

### Step 3 - factory from Hydra config

`get_model_from_cfg(cfg, in_channels, crop_size=...)` is what `scripts/train.py` calls. `model.source: smp`
(the default when the key is absent) goes through `get_model` exactly as before. `custom_unet` reads
`model.custom_unet.*`. The crop size is passed in explicitly (it drives `recommend`/`auto` depth, Step 4);
it also has to be large enough for the chosen depth.
Below, the checks are first tried inline on synthetic configs, then promoted.

In [ ]:
# experiment: the factory's up-front checks, sketched inline on synthetic cfgs (the exported version follows)
from omegaconf import OmegaConf
_c = OmegaConf.create({'model': {'source': 'custom_unet', 'custom_unet': {'encoder_depth': 3, 'base_channels': 8}}, 'train': {'num_classes': 3}})
_cu = _c.model.get('custom_unet', None) or {}
_depth = _cu.get('encoder_depth', 4)
print(_c.model.get('source', 'smp'), _depth, _cu.get('norm', 'group'), 32 >= 2 ** _depth, 4 >= 2 ** _depth)

In [ ]:
# verify: source defaults to smp when absent; a 32px crop fits depth 3, a 4px crop does not; bad depth types are caught
assert OmegaConf.create({'model': {}}).model.get('source', 'smp') == 'smp'
assert 32 >= 2 ** _depth and not 4 >= 2 ** _depth
assert all(isinstance(d, bool) or not isinstance(d, int) or d < 1 for d in (True, 0, -1, 2.5, '4'))

### Step 4 - depth and norm, experiments (#62)

Rule to try: `auto` = `floor(log2(crop / min))` (deepest net whose smallest feature map is still >= `min`, default 32);
`recommend` = the same ratio rounded to the nearest whole step; both clamped to 3..6 downsamplings. An explicit
integer is the user's call and is never clamped. `norm: group | batch | instance` must drop into the same block.

Before promoting anything: what does the rule give for typical crops, what does the clamp do at the extremes,
and do all three norms keep the tensor shape? (Tests for the promoted code sit further down.)

In [ ]:
# experiment: ratio crop/min -> bottleneck steps; floor keeps bottleneck >= min, rounding gets closest to min
import math
for crop in (256, 384, 512, 768, 1024):
    r = math.log2(crop / 32); print(crop, round(r, 2), 'auto', math.floor(r), 'recommend', math.floor(r + 0.5))
for nl in (nn.BatchNorm2d(12), nn.InstanceNorm2d(12, affine=True), nn.GroupNorm(_group_count(12), 12)):
    print(type(nl).__name__, tuple(nl(torch.randn(2, 12, 8, 8)).shape))
print('clamped', [max(3, min(6, math.floor(math.log2(c / 32) + 0.5))) for c in (8, 4096)])

In [ ]:
# verify: 512 -> 4 for both rules; 384 is where they differ (3.58 -> 4 vs floor 3); all norms keep shape
assert math.floor(math.log2(512 / 32)) == 4 and math.floor(math.log2(384 / 32) + 0.5) == 4 and math.floor(math.log2(384 / 32)) == 3
assert all(tuple(nl(torch.randn(2, 12, 8, 8)).shape) == (2, 12, 8, 8)
           for nl in (nn.BatchNorm2d(12), nn.InstanceNorm2d(12, affine=True), nn.GroupNorm(6, 12)))
# clamp: a tiny crop would give 0 downs, a huge one 7; both land inside 3..6
assert [max(3, min(6, math.floor(math.log2(c / 32) + 0.5))) for c in (8, 4096)] == [3, 6]

In [ ]:
#| export
_DEFAULT_ENCODER_DEPTH = 4         # downsamplings when `model.custom_unet.encoder_depth` is absent
_DEFAULT_BOTTLENECK_MIN_SIZE = 32  # smallest bottleneck side `auto` / `recommend` aim for
_MIN_AUTO_DEPTH, _MAX_AUTO_DEPTH = 3, 6  # clamp for `auto` / `recommend` (never applied to an explicit int)

def resolve_encoder_depth(
    depth: Union[int, str],       # 'recommend' | 'auto' | positive int (an int is the user's call and is returned unchanged)
    crop_size: int,               # model-view crop side the net will see
    bottleneck_min_size: int = _DEFAULT_BOTTLENECK_MIN_SIZE # smallest bottleneck side `auto` / `recommend` aim for
) -> int:
    """Number of downsamplings for `model.custom_unet.encoder_depth`.

    `auto` = floor(log2(crop / min)); `recommend` = the same ratio rounded to the nearest step,
    so 512 with min 32 gives 4 (512->256->128->64->32). Both are clamped to [3, 6] (`_MIN_AUTO_DEPTH`, `_MAX_AUTO_DEPTH`); an int is returned as-is.

    Example:
        >>> [resolve_encoder_depth('recommend', c) for c in (64, 512, 4096)]
        [3, 4, 6]
    """
    if isinstance(depth, bool) or not (isinstance(depth, int) or depth in ('recommend', 'auto')):
        raise ValueError(f"custom_unet.encoder_depth must be a positive integer, 'recommend' or 'auto', got {depth!r}")
    if isinstance(depth, int):
        return _positive_int(depth, 'custom_unet.encoder_depth')
    _positive_int(bottleneck_min_size, 'custom_unet.bottleneck_min_size')
    ratio = math.log2(max(crop_size, 1) / bottleneck_min_size)
    steps = math.floor(ratio) if depth == 'auto' else math.floor(ratio + 0.5)
    return max(_MIN_AUTO_DEPTH, min(_MAX_AUTO_DEPTH, steps))

In [ ]:
#| export
def _custom_unet_cfg(model_cfg):
    "The `model.custom_unet` block, or an empty dict when absent or null."
    return model_cfg.get('custom_unet', None) or {}

def _cfg_bottleneck_knobs(custom_cfg):
    "The per-bottleneck knobs of `model.custom_unet`, with defaults for the absent ones."
    return {'aspp_rates': custom_cfg.get('aspp_rates', _DEFAULT_ASPP_RATES),
            'attention_heads': custom_cfg.get('attention_heads', _DEFAULT_ATTENTION_HEADS)}

def _validate_custom_unet_cfg(custom_cfg, crop_size: int):
    "Check `model.custom_unet` before building anything; return `(encoder_depth, norm)`."
    _check_block_style(custom_cfg.get('block_style', _DEFAULT_BLOCK_STYLE), 'custom_unet.block_style')
    bottleneck = custom_cfg.get('bottleneck', 'none')
    _check_bottleneck(bottleneck, 'custom_unet.bottleneck')
    norm = custom_cfg.get('norm', 'group')
    _check_norm(norm, 'custom_unet.norm')
    depth = resolve_encoder_depth(custom_cfg.get('encoder_depth', _DEFAULT_ENCODER_DEPTH), crop_size,
                                 custom_cfg.get('bottleneck_min_size', _DEFAULT_BOTTLENECK_MIN_SIZE))
    _positive_int(depth, 'custom_unet.encoder_depth')
    if crop_size < 2 ** depth:
        raise ValueError(f'crop_size={crop_size} is too small for encoder_depth={depth} (needs >= {2 ** depth})')
    deepest = _channel_plan(custom_cfg.get('base_channels', _DEFAULT_BASE_CHANNELS), custom_cfg.get('max_channels', _DEFAULT_MAX_CHANNELS), depth)[-1]
    _BOTTLENECKS[bottleneck][1](_cfg_bottleneck_knobs(custom_cfg), deepest, crop_size // 2 ** depth, 'custom_unet.')
    if norm == 'batch' and crop_size // 2 ** depth == 1:
        warnings.warn(f'custom_unet.norm=batch with crop_size={crop_size} and encoder_depth={depth} gives a 1x1 bottleneck; '
                      'BatchNorm training with batch size 1 fails there. Use a larger crop, a smaller encoder_depth, or norm=group.',
                      UserWarning)
    return depth, norm

def get_model_from_cfg(
    cfg,                          # Hydra config with `model.*` and `train.num_classes`
    in_channels: int,             # channels fed to the model (from data_mode)
    crop_size: int,               # model-view crop side, passed explicitly by the caller
    pretrained: Optional[bool] = None,   # override `model.pretrained` (e.g. False when rebuilding for export)
    weights_path: Optional[str] = None,  # local encoder .pth (smp only)
) -> nn.Module:
    """Build the model selected by `model.source`: `smp` (default) or `custom_unet`.

    Example:
        >>> m = get_model_from_cfg(cfg, in_channels=1, crop_size=64)  # doctest: +SKIP
    """
    model_cfg = cfg.model
    source = model_cfg.get('source', 'smp')
    if pretrained is None:
        pretrained = bool(model_cfg.get('pretrained', False))
    num_classes = cfg.train.num_classes
    if source == 'smp':
        smp_kwargs = dict(arch=model_cfg.arch, in_channels=in_channels, num_classes=num_classes,
                  pretrained=pretrained, architecture=model_cfg.get('architecture', 'unet'))
        if weights_path is not None:
            smp_kwargs['weights_path'] = weights_path
        return get_model(**smp_kwargs)
    if source == 'custom_unet':
        custom_cfg = _custom_unet_cfg(model_cfg)
        depth, norm = _validate_custom_unet_cfg(custom_cfg, crop_size)
        if pretrained or weights_path is not None:
            warnings.warn('model.source=custom_unet is random-init; pretrained/weights_path are ignored', UserWarning)
        # a pooling step floors odd sizes, so repeated halving gives exactly crop_size // 2**depth
        knobs = _cfg_bottleneck_knobs(custom_cfg)
        return CustomUNet(in_channels=in_channels, num_classes=num_classes, encoder_depth=depth,
                          base_channels=custom_cfg.get('base_channels', _DEFAULT_BASE_CHANNELS),
                          max_channels=custom_cfg.get('max_channels', _DEFAULT_MAX_CHANNELS),
                          norm=norm, num_groups=custom_cfg.get('num_groups', 8),
                          block_style=custom_cfg.get('block_style', _DEFAULT_BLOCK_STYLE),
                          bottleneck=custom_cfg.get('bottleneck', 'none'), bottleneck_size=crop_size // 2 ** depth,
                          aspp_rates=knobs['aspp_rates'], attention_heads=knobs['attention_heads'])
    raise ValueError(f"Unknown model.source {source!r}; choose 'smp' or 'custom_unet'")

A random-init model also needs a different freeze/LR schedule than a pretrained one; that rule lives with the other
shared Hydra resolvers as `resolve_train_schedule` in `15_hydra_pipeline`.

In [ ]:
# test: factory branches on model.source; smp stays the default and unchanged
from omegaconf import OmegaConf
smp_cfg = OmegaConf.create({'model': {'architecture': 'unet', 'arch': 'mobilenet_v2', 'pretrained': False}, 'train': {'num_classes': 1}})
m_smp = get_model_from_cfg(smp_cfg, in_channels=1, crop_size=64)          # no `source` key -> smp
assert isinstance(m_smp, smp.Unet) and not isinstance(m_smp, CustomUNet)
assert tuple(m_smp(torch.randn(1, 1, 64, 64)).shape) == (1, 1, 64, 64)
assert isinstance(get_model_from_cfg(OmegaConf.merge(smp_cfg, {'model': {'source': 'smp'}}), 1, 64), smp.Unet)

cu_cfg = OmegaConf.merge(smp_cfg, {'train': {'num_classes': 3}, 'model': {'source': 'custom_unet',
        'custom_unet': {'encoder_depth': 3, 'base_channels': 8}}})
m_cu = get_model_from_cfg(cu_cfg, in_channels=2, crop_size=32)
assert isinstance(m_cu, CustomUNet) and len(m_cu.encoder.stages) == 4
assert tuple(m_cu(torch.randn(1, 2, 32, 32)).shape) == (1, 3, 32, 32)
for bad, crop in [({'block_style': 'nope'}, 32), ({'bottleneck': 'attention', 'attention_heads': 3}, 32),
                  ({'encoder_depth': 3}, 4)]:                       # depth 3 needs a crop of at least 8
    c2 = OmegaConf.merge(cu_cfg, {'model': {'custom_unet': bad}})
    try: get_model_from_cfg(c2, 2, crop); raise AssertionError(bad)
    except ValueError: pass
try: get_model_from_cfg(OmegaConf.merge(smp_cfg, {'model': {'source': 'nope'}}), 1, 64); raise AssertionError('unknown source')
except ValueError: pass

In [ ]:
# test: encoder_depth and norm are validated up front with clear errors (#61 review)
def _raises(exc, **c):
    try: get_model_from_cfg(OmegaConf.merge(cu_cfg, {'model': {'custom_unet': c}}), 2, 32)
    except exc as e: return str(e)
    raise AssertionError(f'{c} did not raise {exc.__name__}')

for d in (None, '4', True, False, 0, -1, 2.5):
    assert 'encoder_depth' in _raises((ValueError, TypeError), encoder_depth=d), d
assert 'encoder_depth' in _raises(ValueError, encoder_depth='deepest')
assert 'crop_size' in _raises(ValueError, encoder_depth=6)       # 2**6 > 32
for n in ('layer', None):
    assert 'norm' in _raises(ValueError, norm=n), n
# norm is checked before any model is built
import unittest.mock as _mock
with _mock.patch(__name__ + '.CustomUNet', side_effect=AssertionError('built before validation')):
    _raises(ValueError, norm='layer')
# helpers: absent / null / present custom_unet block
assert _custom_unet_cfg(OmegaConf.create({})) == {} and _custom_unet_cfg(OmegaConf.create({'custom_unet': None})) == {}
assert _custom_unet_cfg(OmegaConf.create({'custom_unet': {'norm': 'group'}}))['norm'] == 'group'
assert _validate_custom_unet_cfg({'encoder_depth': 3}, 32) == (3, 'group')

#### Tests for Step 4 (#62)

The promoted `resolve_encoder_depth` is exported above. Footnotes below pin: the depth table, clamps, forcing,
group divisibility over every stage, batch / instance behaviour, and forward shapes.
Note on `norm`: `group` works with any batch size; `batch` is for ablation and behaves differently in train and eval;
`instance` is per-sample.

In [ ]:
# test: resolve_encoder_depth table, clamps, forcing, bad input (#62)
for crop, rec, auto in [(8, 3, 3), (64, 3, 3), (128, 3, 3), (256, 3, 3), (384, 4, 3), (512, 4, 4), (768, 5, 4),
                        (1024, 5, 5), (2048, 6, 6), (4096, 6, 6), (8192, 6, 6)]:
    assert resolve_encoder_depth('recommend', crop) == rec, (crop, 'recommend')
    assert resolve_encoder_depth('auto', crop) == auto, (crop, 'auto')
assert resolve_encoder_depth('recommend', 512, bottleneck_min_size=64) == 3     # 512 -> 64 is 3 downs
assert resolve_encoder_depth('auto', 512, bottleneck_min_size=16) == 5
assert resolve_encoder_depth('auto', 1024, bottleneck_min_size=8) == 6          # 7 clamped down
assert resolve_encoder_depth('auto', 256, bottleneck_min_size=128) == 3         # 1 clamped up
for d in (1, 2, 3, 7, 9):                                                        # integers are honoured, never clamped
    assert resolve_encoder_depth(d, 512) == d
for bad in ('deepest', None, True, 2.5, '4'):
    try: resolve_encoder_depth(bad, 512); raise AssertionError(bad)
    except ValueError as e: assert 'encoder_depth' in str(e)
for bad_min in (0, -4, 2.5, True):
    try: resolve_encoder_depth('auto', 512, bottleneck_min_size=bad_min); raise AssertionError(bad_min)
    except ValueError as e: assert 'bottleneck_min_size' in str(e)

In [ ]:
# test: GroupNorm group count divides EVERY stage's channels, incl. odd base_channels (#62)
for base in (6, 8, 10, 12, 20, 24, 32, 48):
    for want in (8, 32):
        m = CustomUNet(1, 1, encoder_depth=5, base_channels=base, num_groups=want)
        gns = [x for x in m.modules() if isinstance(x, nn.GroupNorm)]
        assert gns and all(g.num_channels % g.num_groups == 0 and g.num_groups <= want for g in gns), (base, want)
        chans = {g.num_channels for g in gns}
        assert {min(base * 2**i, 512) for i in range(6)} <= chans                # every encoder stage has a GroupNorm
        assert all(max(k for k in range(1, want + 1) if c % k == 0) == g.num_groups
                   for g in gns for c in [g.num_channels])                       # and it is the largest divisor <= want

In [ ]:
# test: batch / instance norm build, keep shape, and behave as their names say (#62)
for norm, cls in [('batch', nn.BatchNorm2d), ('instance', nn.InstanceNorm2d)]:
    m = CustomUNet(1, 2, encoder_depth=3, base_channels=8, norm=norm)
    assert any(isinstance(x, cls) for x in m.modules()) and not any(isinstance(x, nn.GroupNorm) for x in m.modules())
    assert tuple(m(torch.randn(2, 1, 32, 32)).shape) == (2, 2, 32, 32)
    assert tuple(m.eval()(torch.randn(1, 1, 37, 41)).shape) == (1, 2, 37, 41)    # odd size in eval
inst = [x for x in CustomUNet(1, 1, 3, 8, norm='instance').modules() if isinstance(x, nn.InstanceNorm2d)]
assert inst and all(x.affine for x in inst)                                      # learnable scale/shift like the others

# BatchNorm: eval uses running stats (batch-independent); train uses the batch (output depends on batch mates).
torch.manual_seed(0)
bn = CustomUNet(1, 1, 3, 8, norm='batch'); xa, xb = torch.randn(1, 1, 32, 32), torch.randn(1, 1, 32, 32)
bn.train(); alone = bn(xa); paired = bn(torch.cat([xa, xb]))[:1]
assert not torch.allclose(alone, paired, atol=1e-5)
bn.eval()
with torch.no_grad(): assert torch.allclose(bn(xa), bn(torch.cat([xa, xb]))[:1], atol=1e-5)
# batch size 1 in train mode works while the bottleneck still has >1 pixel (32 / 2**3 = 4x4); a 1x1 bottleneck would raise
bn.train(); assert tuple(bn(xa).shape) == (1, 1, 32, 32)
for norm in ('group', 'instance'):                                               # batch-independent in train mode too
    g = CustomUNet(1, 1, 3, 8, norm=norm).train()
    assert torch.allclose(g(xa), g(torch.cat([xa, xb]))[:1], atol=1e-5), norm
# one table: every advertised norm builds, and the error lists exactly the table's keys
assert set(_NORMS) == set(_NORM_FACTORIES) == {'group', 'batch', 'instance'}
assert all(isinstance(_norm_layer(nm, 12, 8), nn.Module) for nm in _NORMS)
try: _norm_layer('layer', 12, 8); raise AssertionError('unknown norm accepted')
except ValueError as e: assert "['group', 'batch', 'instance']" in str(e)

In [ ]:
# test: factory resolves recommend / auto from the crop it is given, and forwards bottleneck_min_size / norm (#62)
def _depth_of(crop, **c):
    cfg = OmegaConf.merge(cu_cfg, {'model': {'custom_unet': {'base_channels': 4, **c}}})
    m = get_model_from_cfg(cfg, 2, crop); return len(m.encoder.stages) - 1, m
assert _depth_of(512, encoder_depth='recommend')[0] == 4
assert _depth_of(512, encoder_depth='auto')[0] == 4
assert _depth_of(384, encoder_depth='recommend')[0] == 4 and _depth_of(384, encoder_depth='auto')[0] == 3
assert _depth_of(256, encoder_depth='auto', bottleneck_min_size=16)[0] == 4
assert _depth_of(64, encoder_depth='recommend')[0] == 3                           # clamped up
assert _depth_of(4096, encoder_depth='recommend')[0] == 6                         # clamped down
assert _depth_of(512, encoder_depth=2)[0] == 2                                    # forced
d, m = _depth_of(64, encoder_depth='recommend', norm='instance')
assert d == 3 and tuple(m(torch.randn(1, 2, 64, 64)).shape) == (1, 3, 64, 64)
d_bn, m_bn = _depth_of(64, encoder_depth='recommend', norm='batch')
assert d_bn == 3 and len(m_bn.encoder.stages) == 4                                # recommend(64) -> 3 downs -> 4 stages
assert any(isinstance(x, nn.BatchNorm2d) for x in m_bn.modules()) and not any(isinstance(x, nn.GroupNorm) for x in m_bn.modules())
assert 'bottleneck_min_size' in _raises(ValueError, encoder_depth='auto', bottleneck_min_size=0)
assert _validate_custom_unet_cfg({'encoder_depth': 'auto', 'bottleneck_min_size': 16}, 256) == (4, 'group')
assert _validate_custom_unet_cfg({'encoder_depth': 'recommend', 'norm': 'batch'}, 512) == (4, 'batch')
try: _validate_custom_unet_cfg({'encoder_depth': 'recommend'}, 4); raise AssertionError('tiny crop')
except ValueError as e: assert 'crop_size' in str(e)

# test: BatchNorm + 1x1 bottleneck warns (batch size 1 cannot train there); no warning otherwise (#62 review)
import warnings as _w
def _warned(cfg, crop):
    with _w.catch_warnings(record=True) as rec:
        _w.simplefilter('always'); out = _validate_custom_unet_cfg(cfg, crop)
    return out, [str(r.message) for r in rec if issubclass(r.category, UserWarning)]
out, msgs = _warned({'encoder_depth': 5, 'norm': 'batch'}, 32)                      # 32 / 2**5 = 1
assert out == (5, 'batch') and len(msgs) == 1 and 'norm=group' in msgs[0] and '1x1' in msgs[0]
assert _warned({'encoder_depth': 4, 'norm': 'batch'}, 128)[1] == []                 # 8x8 bottleneck
assert _warned({'encoder_depth': 5, 'norm': 'group'}, 32)[1] == []                  # only BatchNorm is affected
assert _warned({'encoder_depth': 5, 'norm': 'instance'}, 32)[1] == []


In [ ]:
# test: CUSTOM_UNET_TRAIN_SMOKE one-epoch train() on synthetic data with the custom model, frozen-encoder warmup included
import tempfile
from torch.utils.data import Dataset, DataLoader
from segmentation_tools_pt.train import train

class _SynthSeg(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        g = torch.Generator().manual_seed(i)
        return {'image': torch.rand(1, 32, 32, generator=g), 'mask': (torch.rand(32, 32, generator=g) > 0.5).long(), 'sample_id': str(i)}

_ld = DataLoader(_SynthSeg(), batch_size=2)
for _style in ('double', 'residual'):                                           # same smoke, both block styles (#63)
    _m = get_model_from_cfg(OmegaConf.merge(cu_cfg, {'train': {'num_classes': 1}, 'model': {'custom_unet': {'block_style': _style}}}),
                            in_channels=1, crop_size=32)
    with tempfile.TemporaryDirectory() as d:
        hist = train(root=None, model=_m, train_loader=_ld, val_loader=_ld, num_classes=1, num_epochs=1, batch_size=2,
                     device_str='cpu', checkpoint_dir=d, use_amp=False, resume=False, freeze_epochs=1, num_workers=0)
    assert len(hist) == 1 and 'val_iou' in hist[0], _style

In [ ]:
# test: CUSTOM_UNET_ONNX_SMOKE FP32 export of the custom model (skips cleanly without onnx / onnxruntime)
try:
    import onnx, onnxruntime
except ImportError:
    print('skipped: onnx / onnxruntime not installed')
else:
    import tempfile
    from pathlib import Path
    from segmentation_tools_pt.export import export_onnx
    _cm = CustomUNet(1, 3, encoder_depth=3, base_channels=8).eval()
    with tempfile.TemporaryDirectory() as d:
        out = Path(d) / 'custom_unet.onnx'
        export_onnx(_cm, str(out), in_channels=1, input_size=32, opset=18, verify=True)
        assert out.exists() and out.stat().st_size > 0
    # dynamic H/W: the exported graph must also run on other even sizes and on odd sizes (decoder resize is traced)
    import numpy as np
    with tempfile.TemporaryDirectory() as d:
        dyn = str(Path(d) / 'dyn.onnx')
        torch.onnx.export(_cm, torch.randn(1, 1, 32, 32), dyn, input_names=['image'], output_names=['logits'], opset_version=18, dynamo=False,
                          dynamic_axes={'image': {0: 'b', 2: 'h', 3: 'w'}, 'logits': {0: 'b', 2: 'h', 3: 'w'}})
        sess = onnxruntime.InferenceSession(dyn)
        for hw in [(32, 32), (48, 48), (37, 41), (65, 65)]:
            xi = torch.randn(1, 1, *hw)
            with torch.no_grad(): ref = _cm(xi).numpy()
            assert np.abs(sess.run(None, {'image': xi.numpy()})[0] - ref).max() < 1e-4, hw


In [ ]:
# test: CUSTOM_UNET_ONNX_NORM_SMOKE batch / instance norm and residual blocks export (#63) with dynamic H/W and match eager eval (#62)
try:
    import onnx, onnxruntime
except ImportError:
    print('skipped: onnx / onnxruntime not installed')
else:
    import numpy as np, tempfile
    from pathlib import Path
    for norm, style in [('batch', 'double'), ('instance', 'double'), ('group', 'residual'), ('batch', 'residual'), ('instance', 'residual')]:
        _cm = CustomUNet(1, 2, encoder_depth=3, base_channels=8, norm=norm, block_style=style).eval()
        with tempfile.TemporaryDirectory() as d:
            dyn = str(Path(d) / f'{norm}_{style}.onnx')
            torch.onnx.export(_cm, torch.randn(1, 1, 32, 32), dyn, input_names=['image'], output_names=['logits'], opset_version=18, dynamo=False,
                              dynamic_axes={'image': {0: 'b', 2: 'h', 3: 'w'}, 'logits': {0: 'b', 2: 'h', 3: 'w'}})
            sess = onnxruntime.InferenceSession(dyn)
            for hw in [(32, 32), (48, 48), (37, 41)]:
                xi = torch.randn(1, 1, *hw)
                with torch.no_grad(): ref = _cm(xi).numpy()
                assert np.abs(sess.run(None, {'image': xi.numpy()})[0] - ref).max() < 1e-4, (norm, style, hw)

In [68]:
#| hide
import nbdev; nbdev.nbdev_export()

#### Tests for Step 5 (#63)

Footnotes for the residual block: shapes with every norm (odd sizes too), the single block table and its
early error, parameter count (residual adds only the 1x1 projections), and that the skip really carries signal.

In [ ]:
# test: block_style=residual keeps the shape with every norm, odd sizes, and builds residual blocks in encoder AND decoder (#63)
for k in (1, 3):
    assert tuple(CustomUNet(1, k, 4, block_style='residual')(torch.randn(1, 1, 64, 64)).shape) == (1, k, 64, 64)
for norm in ('group', 'batch', 'instance'):
    m = CustomUNet(1, 2, encoder_depth=3, base_channels=8, norm=norm, block_style='residual')
    assert tuple(m(torch.randn(2, 1, 32, 32)).shape) == (2, 2, 32, 32)
    assert tuple(m.eval()(torch.randn(1, 1, 37, 41)).shape) == (1, 2, 37, 41)
blocks = [x for x in m.modules() if isinstance(x, (_DoubleConv, _ResidualBlock))]
assert len(blocks) == 4 + 3 and all(isinstance(b, _ResidualBlock) for b in blocks)      # 4 encoder stages + 3 decoder blocks
assert all(isinstance(b, _DoubleConv) for b in [x for x in CustomUNet(1, 1, 3, 8).modules() if isinstance(x, (_DoubleConv, _ResidualBlock))])
assert hasattr(m, 'encoder') and hasattr(m, 'decoder') and hasattr(m, 'segmentation_head')
assert m.segmentation_head.out_channels == 2
# one table for block_style, like norm; the error lists exactly the table's keys and is raised before anything is built
assert set(_BLOCK_STYLES) == set(_BLOCK_CLASSES) == {'double', 'residual'} and _BLOCK_CLASSES['residual'] is _ResidualBlock
try: CustomUNet(block_style='dense'); raise AssertionError('unknown block_style accepted')
except ValueError as e: assert "['double', 'residual']" in str(e) and 'block_style' in str(e)
with _mock.patch(__name__ + '.CustomUNet', side_effect=AssertionError('built before validation')):
    assert "['double', 'residual']" in _raises(ValueError, block_style='dense')
    for bs in (None, 1):
        _raises(ValueError, block_style=bs)
assert _validate_custom_unet_cfg({'encoder_depth': 3, 'block_style': 'residual'}, 32) == (3, 'group')

# test: residual has exactly the 1x1 projections more parameters than double (#63)
def _nparams(mod): return sum(p.numel() for p in mod.parameters())
_dbl, _res = CustomUNet(1, 3, 3, base_channels=8), CustomUNet(1, 3, 3, base_channels=8, block_style='residual')
# channels 8,16,32,64: encoder projections 1->8, 8->16, 16->32, 32->64; decoder (2*ch -> ch) for ch = 32, 16, 8 (no bias)
assert _nparams(_res) - _nparams(_dbl) == (1*8 + 8*16 + 16*32 + 32*64) + (64*32 + 32*16 + 16*8) == 5384
assert _nparams(_res.encoder) - _nparams(_dbl.encoder) == 2696 and _nparams(_res.decoder) - _nparams(_dbl.decoder) == 2688

# test: the skip carries signal, so the add is not dead code (#63)
torch.manual_seed(0)
def _kill_residual_path(blk):
    for p in (blk.conv2.weight, blk.norm2.weight, blk.norm2.bias): nn.init.zeros_(p)
_xi = torch.randn(2, 8, 16, 16)
_same = _ResidualBlock(8, 8, 'group', 8); _kill_residual_path(_same)
assert torch.allclose(_same(_xi), torch.relu(_xi), atol=1e-6)                           # identity skip: out = relu(x)
_proj = _ResidualBlock(4, 8, 'group', 8); _kill_residual_path(_proj)
_xp = torch.randn(2, 4, 16, 16)
assert torch.allclose(_proj(_xp), torch.relu(_proj.skip(_xp)), atol=1e-6) and isinstance(_proj.skip, nn.Conv2d)
assert isinstance(_same.skip, nn.Identity) and _proj.skip.kernel_size == (1, 1)
# and the other way round: with the skip removed the output changes, in a model too
_r = _ResidualBlock(8, 8, 'group', 8).eval(); _out = _r(_xi)
_r.skip = nn.Module(); _r.skip.forward = lambda t: torch.zeros_like(t)
assert not torch.allclose(_out, _r(_xi), atol=1e-4)
_gm = CustomUNet(1, 1, 3, 8, block_style='residual'); _gx = torch.randn(1, 1, 32, 32, requires_grad=True)
for b in _gm.modules():
    if isinstance(b, _ResidualBlock): _kill_residual_path(b)
_gm(_gx).sum().backward()
assert _gx.grad.abs().sum() > 0                                                          # signal still flows with every conv path off

#### Tests for Step 6 (#64)

Footnotes for the ASPP bottleneck: the module keeps size and channels (every norm, odd sizes), the trim table, the
default `none` being a true identity (same parameters, same output), config checks before anything is built, every
parameter reaching an optimizer group, gradients reaching every branch, and ONNX export with dynamic H/W.

In [ ]:
# test: ASPP keeps (B, C, H, W) for every norm, odd sizes and tiny maps; branch count = rates + pooling (#64)
for norm in ('group', 'batch', 'instance'):
    a = _ASPP(32, [1, 6, 12], norm, 8)
    assert len(a.branches) == 3
    for hw in [(16, 16), (17, 23), (4, 4)]:
        assert tuple(a(torch.randn(2, 32, *hw)).shape) == (2, 32, *hw), (norm, hw)
    assert tuple(a.eval()(torch.randn(1, 32, 2, 2)).shape) == (1, 32, 2, 2)             # batch 1: the pooled branch (one pixel per channel) has no norm
assert tuple(_ASPP(6, [1, 6], 'group', 8)(torch.randn(1, 6, 8, 8)).shape) == (1, 6, 8, 8)    # C // 4 = 1 branch channel still works

# test: trim_aspp_rates table (#64)
R = [1, 6, 12, 18]
assert trim_aspp_rates(R, 64) == trim_aspp_rates(R, 16) == R
assert trim_aspp_rates(R, 15) == [1, 6, 12] and trim_aspp_rates(R, 13) == [1, 6, 12] and trim_aspp_rates(R, 12) == [1, 6]
assert trim_aspp_rates(R, 8) == [1, 6] and trim_aspp_rates(R, 6) == [1] and trim_aspp_rates(R, 1) == [1]
assert trim_aspp_rates([6, 12], 4) == [1] and trim_aspp_rates((18, 1), 16) == [18, 1]        # nothing fits -> plain 3x3; user order and tuples are kept

# test: full UNet with aspp keeps the output shape and puts the module between encoder and decoder (#64)
for style in ('double', 'residual'):
    m = CustomUNet(1, 3, encoder_depth=3, base_channels=8, bottleneck='aspp', aspp_rates=[1, 2], block_style=style)
    assert isinstance(m.bottleneck, _ASPP)
    assert tuple(m(torch.randn(2, 1, 32, 32)).shape) == (2, 3, 32, 32)
    assert tuple(m.eval()(torch.randn(1, 1, 37, 41)).shape) == (1, 3, 37, 41)
assert m.bottleneck.project[0].out_channels == 64

# test: bottleneck='none' is the default and an identity: no new parameters, same weights -> same output (#64)
torch.manual_seed(0); d0 = CustomUNet(1, 2, 3, base_channels=8)
torch.manual_seed(0); d1 = CustomUNet(1, 2, 3, base_channels=8, bottleneck='none')
assert isinstance(d0.bottleneck, nn.Identity) and not any(k.startswith('bottleneck') for k in d0.state_dict())
assert list(d0.state_dict()) == list(d1.state_dict())
_x = torch.randn(1, 1, 32, 32)
assert torch.equal(d0.eval()(_x), d1.eval()(_x))
assert torch.equal(d0.bottleneck(_x), _x)

# test: config checks come before any model is built (#64)
assert 'aspp' in _raises(ValueError, bottleneck='dense') and 'bottleneck' in _raises(ValueError, bottleneck='dense')
for b in (None, 1):
    _raises(ValueError, bottleneck=b)
for bad in ([], [0], [-1, 2], [1.5], [True], 'x', 6, None):
    assert 'aspp_rates' in _raises(ValueError, bottleneck='aspp', aspp_rates=bad), bad
with _mock.patch(__name__ + '.CustomUNet', side_effect=AssertionError('built before validation')):
    _raises(ValueError, bottleneck='dense')
    _raises(ValueError, bottleneck='aspp', aspp_rates=[])
assert _validate_custom_unet_cfg({'encoder_depth': 3, 'bottleneck': 'aspp'}, 32) == (3, 'group')
assert _validate_custom_unet_cfg({'encoder_depth': 3, 'bottleneck': 'none', 'aspp_rates': []}, 32) == (3, 'group')   # unused knob is not checked
assert isinstance(get_model_from_cfg(OmegaConf.merge(cu_cfg, {'model': {'custom_unet': {'aspp_rates': 'x'}}}), 1, 32).bottleneck, nn.Identity)
try: CustomUNet(bottleneck='dense'); raise AssertionError('unknown bottleneck accepted')
except ValueError as e: assert 'bottleneck' in str(e) and "'aspp'" in str(e)

# test: the factory builds aspp from config and trims rates to the real bottleneck side (crop // 2**depth) (#64)
def _aspp_model(crop, depth, **c):
    cfg = OmegaConf.merge(cu_cfg, {'model': {'custom_unet': {'bottleneck': 'aspp', 'encoder_depth': depth, 'base_channels': 8, **c}}})
    return get_model_from_cfg(cfg, 1, crop)
assert len(_aspp_model(512, 4).bottleneck.branches) == 4
assert len(_aspp_model(256, 4).bottleneck.branches) == 4
assert [b[0].dilation[0] for b in _aspp_model(128, 4).bottleneck.branches] == [1, 6]
assert [b[0].dilation[0] for b in _aspp_model(64, 3, aspp_rates=[1, 3, 9]).bottleneck.branches] == [1, 3]   # side 8: 9 dropped
assert [b[0].dilation[0] for b in _aspp_model(32, 4, aspp_rates=[6, 12]).bottleneck.branches] == [1]
assert tuple(_aspp_model(64, 3)(torch.randn(1, 1, 64, 64)).shape) == (1, cu_cfg.train.num_classes, 64, 64)
assert isinstance(get_model_from_cfg(cu_cfg, 1, 32).bottleneck, nn.Identity)

# test: gradients reach every weight matrix of the bottleneck (#64); optimizer-group coverage is tested for all bottlenecks under #97
def _grads_reach_bottleneck(**kw):
    am = CustomUNet(1, 1, 3, base_channels=8, **kw)
    am.zero_grad(); am(torch.randn(2, 1, 32, 32)).sum().backward()
    assert all(p.grad is not None and p.grad.abs().sum() > 0 for p in am.bottleneck.parameters() if p.dim() > 1), kw
_grads_reach_bottleneck(bottleneck='aspp')

# test: CustomUNet trims its own rates when told the bottleneck side; without it the rates are used as given (#64)
def _rates(m): return [b[0].dilation[0] for b in m.bottleneck.branches]
assert _rates(CustomUNet(1, 1, 3, 8, bottleneck='aspp')) == [1, 6, 12, 18]                        # no size: as given
assert _rates(CustomUNet(1, 1, 3, 8, bottleneck='aspp', bottleneck_size=8)) == [1, 6]
assert _rates(CustomUNet(1, 1, 3, 8, bottleneck='aspp', bottleneck_size=32)) == [1, 6, 12, 18]
assert _rates(CustomUNet(1, 1, 3, 8, bottleneck='aspp', aspp_rates=[6, 12], bottleneck_size=4)) == [1]
assert isinstance(CustomUNet(1, 1, 3, 8, bottleneck_size=4).bottleneck, nn.Identity)             # size is ignored for none
for bad in (0, -1, 1.5, True):
    try: CustomUNet(1, 1, 3, 8, bottleneck='aspp', bottleneck_size=bad); raise AssertionError(bad)
    except ValueError as e: assert 'bottleneck_size' in str(e)

In [ ]:
# test: CUSTOM_UNET_ONNX_ASPP_SMOKE aspp bottleneck exports with dynamic H/W and matches eager eval (skips cleanly without onnx) (#64)
try:
    import onnx, onnxruntime
except ImportError:
    print('skipped: onnx / onnxruntime not installed')
else:
    import numpy as np, tempfile
    from pathlib import Path
    for norm in ('group', 'batch', 'instance'):
        _cm = CustomUNet(1, 2, encoder_depth=3, base_channels=8, norm=norm, bottleneck='aspp', aspp_rates=[1, 2, 4]).eval()
        with tempfile.TemporaryDirectory() as d:
            dyn = str(Path(d) / f'aspp_{norm}.onnx')
            torch.onnx.export(_cm, torch.randn(1, 1, 32, 32), dyn, input_names=['image'], output_names=['logits'], opset_version=18, dynamo=False,
                              dynamic_axes={'image': {0: 'b', 2: 'h', 3: 'w'}, 'logits': {0: 'b', 2: 'h', 3: 'w'}})
            sess = onnxruntime.InferenceSession(dyn)
            for hw in [(32, 32), (48, 48), (37, 41)]:
                xi = torch.randn(1, 1, *hw)
                with torch.no_grad(): ref = _cm(xi).numpy()
                assert np.abs(sess.run(None, {'image': xi.numpy()})[0] - ref).max() < 1e-4, (norm, hw)

#### Tests for Step 7 (#65)

Footnotes for the attention bottleneck: size and channels kept (odd sizes), the whole UNet with every block style and norm,
heads must divide the channel count (checked before anything is built), `attention` and `aspp` as separate values, `none`
untouched, optimizer groups and gradients, positions really mixing, and ONNX export (fixed size and dynamic H/W).

In [ ]:
# test: attention keeps (B, C, H, W) for odd, non-square and 1x1 maps, and mixes spatial positions (#65)
for hw in [(8, 8), (5, 7), (17, 23), (1, 1)]:
    at = _AttentionBottleneck(32, 4).eval()
    assert tuple(at(torch.randn(2, 32, *hw)).shape) == (2, 32, *hw), hw
at = _AttentionBottleneck(32, 4).eval()
_a = torch.randn(1, 32, 6, 6); _b = _a.clone(); _b[..., 0, 0] += 1
with torch.no_grad(): _d = (at(_a) - at(_b)).abs()
assert _d[..., 5, 5].max() > 0 and _d[..., 3, 2].max() > 0          # far pixels move too: positions mix across the whole map, not just a neighbourhood
assert tuple(_AttentionBottleneck(6, 3).eval()(torch.randn(1, 6, 4, 4)).shape) == (1, 6, 4, 4)

In [ ]:
# test: full UNet with attention keeps the output shape for every block style and norm (#65)
for style in ('double', 'residual'):
    for norm in ('group', 'batch', 'instance'):
        m = CustomUNet(1, 3, encoder_depth=3, base_channels=8, norm=norm, block_style=style, bottleneck='attention', attention_heads=2)
        assert isinstance(m.bottleneck, _AttentionBottleneck)
        assert tuple(m(torch.randn(2, 1, 32, 32)).shape) == (2, 3, 32, 32), (style, norm)
        assert tuple(m.eval()(torch.randn(1, 1, 37, 41)).shape) == (1, 3, 37, 41), (style, norm)
assert CustomUNet(1, 1, 3, 8, bottleneck='attention').bottleneck.heads == 4

In [ ]:
# test: attention_heads must divide the bottleneck channels, named in the error, checked before any model is built (#65)
for bad in (3, 0, -1, 1.5, True, '4', None, 128):
    assert 'custom_unet.attention_heads' in _raises(ValueError, bottleneck='attention', attention_heads=bad), bad
with _mock.patch(__name__ + '.CustomUNet', side_effect=AssertionError('built before validation')):
    _raises(ValueError, bottleneck='attention', attention_heads=3)
for heads in (1, 2, 4, 8, 64):
    assert isinstance(get_model_from_cfg(OmegaConf.merge(cu_cfg, {'model': {'custom_unet': {'bottleneck': 'attention', 'attention_heads': heads}}}), 1, 32).bottleneck, _AttentionBottleneck)
assert _validate_custom_unet_cfg({'encoder_depth': 3, 'base_channels': 8, 'bottleneck': 'attention'}, 32) == (3, 'group')
assert _validate_custom_unet_cfg({'encoder_depth': 3, 'base_channels': 8, 'max_channels': 16, 'bottleneck': 'attention', 'attention_heads': 16}, 32) == (3, 'group')   # max_channels caps the deepest width at 16, so 16 heads fit although base*2**depth is 64
assert 'custom_unet.attention_heads' in _raises(ValueError, bottleneck='attention', max_channels=12, attention_heads=8)         # 8 divides the uncapped 64 but not the capped 12, so the validator must use the capped width
try: CustomUNet(1, 1, 3, 8, bottleneck='attention', attention_heads=3); raise AssertionError('heads=3 on 64 channels accepted')
except ValueError as e: assert 'attention_heads' in str(e)

In [ ]:
# test: one bottleneck value at a time; unused knobs are ignored, not validated (#65)
assert isinstance(CustomUNet(1, 1, 3, 8, bottleneck='aspp', attention_heads=3).bottleneck, _ASPP)
assert isinstance(CustomUNet(1, 1, 3, 8, bottleneck='attention', aspp_rates=[]).bottleneck, _AttentionBottleneck)
assert isinstance(get_model_from_cfg(OmegaConf.merge(cu_cfg, {'model': {'custom_unet': {'bottleneck': 'attention', 'aspp_rates': 'x'}}}), 1, 32).bottleneck, _AttentionBottleneck)
assert isinstance(get_model_from_cfg(OmegaConf.merge(cu_cfg, {'model': {'custom_unet': {'bottleneck': 'aspp', 'attention_heads': 3}}}), 1, 32).bottleneck, _ASPP)
assert isinstance(get_model_from_cfg(OmegaConf.merge(cu_cfg, {'model': {'custom_unet': {'bottleneck': 'none', 'attention_heads': 3}}}), 1, 32).bottleneck, nn.Identity)
assert 'attention' in _raises(ValueError, bottleneck='both')

In [ ]:
# test: attention adds bottleneck parameters (#65); the default adding none is tested under #64
assert any(k.startswith('bottleneck') for k in CustomUNet(1, 2, 3, 8, bottleneck='attention').state_dict())

In [ ]:
# test: gradients reach every weight matrix of the attention layer (#65)
_grads_reach_bottleneck(bottleneck='attention', attention_heads=2)

In [ ]:
# test: the validator and CustomUNet agree on the deepest channel count, because both take it from `_channel_plan` (#65 review)
def _raises_cfg(custom_cfg, crop_size):
    try: _validate_custom_unet_cfg(custom_cfg, crop_size)
    except ValueError as e: return str(e)
    raise AssertionError(f'{custom_cfg} did not raise')
for base, cap, depth in [(8, 512, 3), (8, 16, 3), (32, 512, 4), (32, 100, 4), (6, 12, 2)]:
    deepest = _channel_plan(base, cap, depth)[-1]
    assert CustomUNet(1, 1, depth, base, cap, bottleneck='attention', attention_heads=1).bottleneck.qkv.in_features == deepest
    cfg = {'encoder_depth': depth, 'base_channels': base, 'max_channels': cap, 'bottleneck': 'attention'}
    assert _validate_custom_unet_cfg({**cfg, 'attention_heads': deepest}, 2 ** depth)[0] == depth       # heads == channels is the largest that divides
    assert 'attention_heads' in _raises_cfg(cfg | {'attention_heads': deepest + 1}, 2 ** depth), (base, cap, depth)

In [ ]:
# test: CUSTOM_UNET_ONNX_ATTENTION_SMOKE attention bottleneck: fixed-size and dynamic H/W FP32 export match eager eval (skips cleanly without onnx) (#65)
try:
    import onnx, onnxruntime
except ImportError:
    print('skipped: onnx / onnxruntime not installed')
else:
    import numpy as np, tempfile
    from pathlib import Path
    _cm = CustomUNet(1, 2, encoder_depth=3, base_channels=8, bottleneck='attention', attention_heads=2).eval()
    with tempfile.TemporaryDirectory() as d:
        fixed, dyn = str(Path(d) / 'att_fixed.onnx'), str(Path(d) / 'att_dyn.onnx')
        torch.onnx.export(_cm, torch.randn(1, 1, 32, 32), fixed, input_names=['image'], output_names=['logits'], opset_version=18, dynamo=False)
        torch.onnx.export(_cm, torch.randn(1, 1, 32, 32), dyn, input_names=['image'], output_names=['logits'], opset_version=18, dynamo=False,
                          dynamic_axes={'image': {0: 'b', 2: 'h', 3: 'w'}, 'logits': {0: 'b', 2: 'h', 3: 'w'}})
        onnx.checker.check_model(onnx.load(fixed)); onnx.checker.check_model(onnx.load(dyn))
        xi = torch.randn(1, 1, 32, 32)
        with torch.no_grad(): ref = _cm(xi).numpy()
        assert np.abs(onnxruntime.InferenceSession(fixed).run(None, {'image': xi.numpy()})[0] - ref).max() < 1e-4
        sess = onnxruntime.InferenceSession(dyn)
        for hw in [(32, 32), (48, 48), (37, 41)]:                          # dynamic H/W works because attention is written with SDPA, not nn.MultiheadAttention
            xi = torch.randn(1, 1, *hw)
            with torch.no_grad(): ref = _cm(xi).numpy()
            assert np.abs(sess.run(None, {'image': xi.numpy()})[0] - ref).max() < 1e-4, hw

In [ ]:
# test: why the layer avoids nn.MultiheadAttention: its dynamic-H/W ONNX graph only runs at the traced size (reproduced with legacy exporter, opset 18) (#65 review)
try:
    import onnx, onnxruntime
except ImportError:
    print('skipped: onnx / onnxruntime not installed')
else:
    import numpy as np, tempfile
    from pathlib import Path
    class _MhaMap(nn.Module):
        def __init__(self):
            super().__init__(); self.attn = nn.MultiheadAttention(16, 2, batch_first=True)
        def forward(self, x):
            t = x.flatten(2).transpose(1, 2)
            return (t + self.attn(t, t, t, need_weights=False)[0]).transpose(1, 2).reshape(x.shape)
    _mm = _MhaMap().eval()
    with tempfile.TemporaryDirectory() as d:
        p = str(Path(d) / 'mha_dyn.onnx')
        torch.onnx.export(_mm, torch.randn(1, 16, 4, 4), p, input_names=['x'], output_names=['y'], opset_version=18, dynamo=False,
                          dynamic_axes={'x': {0: 'b', 2: 'h', 3: 'w'}, 'y': {0: 'b', 2: 'h', 3: 'w'}})
        sess = onnxruntime.InferenceSession(p)
        xi = torch.randn(1, 16, 4, 4)
        with torch.no_grad(): ref = _mm(xi).numpy()
        assert np.abs(sess.run(None, {'x': xi.numpy()})[0] - ref).max() < 1e-4         # exports and runs at the traced size
        try: sess.run(None, {'x': torch.randn(1, 16, 6, 6).numpy()}); raise AssertionError('MultiheadAttention ran at another size')
        except onnxruntime.capi.onnxruntime_pybind11_state.Fail: pass                    # the token count is baked into a Reshape

In [ ]:
# test: an attention model goes through the repo's `export_onnx` seam with verification (skips cleanly without onnx / onnxruntime) (#65 review)
try:
    import onnx, onnxruntime
except ImportError:
    print('skipped: onnx / onnxruntime not installed')
else:
    import tempfile
    from pathlib import Path
    from segmentation_tools_pt.export import export_onnx
    _am = CustomUNet(1, 2, encoder_depth=3, base_channels=8, bottleneck='attention', attention_heads=2).eval()
    with tempfile.TemporaryDirectory() as d:
        out = Path(d) / 'attention.onnx'
        export_onnx(_am, str(out), in_channels=1, input_size=32, opset=18, verify=True)
        assert out.exists() and out.stat().st_size > 0

#### Tests for the documented commands (#66)

`docs/TRAINING_CLI.md`, the `choose-model-architecture` skill and the commented `custom_unet` blocks in the two shipped
configs show `model.*` overrides. A doc that does not run is worse than none, so this cell composes each documented
override from the real YAML files and builds the model from it.

In [ ]:
# test: every documented custom_unet command composes from the shipped configs and builds the model (#66)
import warnings
from pathlib import Path
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from segmentation_tools_pt.hydra_pipeline import resolve_crop_size

_cfg_dir = next(str(p.resolve()) for p in (Path.cwd() / 'configs', Path.cwd().parent / 'configs') if p.is_dir())
_CU = '+model.custom_unet.'
documented = [   # (config, overrides, bottleneck class the build must contain)
    ('train', ['model.source=custom_unet', _CU + 'encoder_depth=recommend', _CU + 'norm=group', 'model.pretrained=false', 'model.weights_path=null'], nn.Identity),
    ('train', ['model.source=custom_unet', _CU + 'block_style=residual', _CU + 'norm=instance'], nn.Identity),
    ('train', ['model.source=custom_unet', _CU + 'bottleneck=aspp'], _ASPP),
    ('train', ['model.source=custom_unet', _CU + 'bottleneck=attention', _CU + 'attention_heads=4'], _AttentionBottleneck),
    ('train_multilabel_3head', ['model.source=custom_unet', 'model.pretrained=false'], nn.Identity),
    ('train_multilabel_3head', ['model.source=custom_unet', _CU + 'bottleneck=aspp'], _ASPP),
]
GlobalHydra.instance().clear()
for name, overrides, bottleneck_cls in documented:
    with initialize_config_dir(config_dir=_cfg_dir, version_base='1.3'):
        cfg = compose(config_name=name, overrides=overrides)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')               # random-init warns when the preset says pretrained
        m = get_model_from_cfg(cfg, in_channels=1, crop_size=resolve_crop_size(cfg))
    assert isinstance(m, CustomUNet) and isinstance(m.bottleneck, bottleneck_cls), (name, overrides)
    GlobalHydra.instance().clear()
# the default of both configs stays the SMP model
for name in ('train', 'train_multilabel_3head'):
    with initialize_config_dir(config_dir=_cfg_dir, version_base='1.3'):
        assert compose(config_name=name).model.get('source', 'smp') == 'smp', name
    GlobalHydra.instance().clear()

#### Tests for the clean-ups (#97)

Footnotes for the follow-up clean-ups: optimizer groups, the single bottleneck table, config errors for unhashable
values and the named channel defaults.

In [ ]:
# test: get_param_groups asks the model for its decoder-side parameters; groups are unchanged for every model (#97)
def _ids(ps): return [id(p) for p in ps]
for bn in ('none', 'aspp', 'attention'):
    m = CustomUNet(1, 2, 3, base_channels=8, bottleneck=bn)
    g = get_param_groups(m, encoder_lr=1e-4, decoder_lr=1e-3)
    assert [x['lr'] for x in g] == [1e-4, 1e-3]
    assert _ids(g[0]['params']) == _ids(m.encoder.parameters())
    assert _ids(g[1]['params']) == _ids(m.decoder.parameters()) + _ids(m.segmentation_head.parameters()) + _ids(m.bottleneck.parameters())
    assert _ids(g[1]['params']) == _ids(m.decoder_side_parameters())
    all_ids = _ids(g[0]['params']) + _ids(g[1]['params'])
    assert len(all_ids) == len(set(all_ids)) == len(list(m.parameters()))

class _WithExtra(nn.Module):                     # a layout get_param_groups has never heard of
    def __init__(self):
        super().__init__()
        self.encoder, self.extra, self.decoder, self.segmentation_head = nn.Linear(2, 2), nn.Linear(2, 2), nn.Linear(2, 2), nn.Linear(2, 1)
    def decoder_side_parameters(self):
        return [*self.decoder.parameters(), *self.extra.parameters(), *self.segmentation_head.parameters()]
_we = _WithExtra(); _g = get_param_groups(_we)
assert len(_g[1]['params']) == 6 and set(_ids(_we.extra.parameters())) <= set(_ids(_g[1]['params']))   # extra trains at the decoder rate

_smp = get_model('mobilenet_v2', in_channels=1, num_classes=1, pretrained=False)
_g = get_param_groups(_smp)
assert _ids(_g[0]['params']) == _ids(_smp.encoder.parameters())
assert _ids(_g[1]['params']) == _ids(_smp.decoder.parameters()) + _ids(_smp.segmentation_head.parameters())
assert len(get_param_groups(nn.Linear(2, 2))) == 1                                              # no .encoder: one group
freeze_encoder(m)
assert all(p.requires_grad for p in m.bottleneck.parameters()) and not any(p.requires_grad for p in m.encoder.parameters())
unfreeze_encoder(m)

In [ ]:
# test: list / dict / None for a choice key is a ValueError naming the key, never a TypeError (#97)
for key in ('bottleneck', 'norm', 'block_style'):
    for bad in (['aspp', 'attention'], {'a': 1}, None):
        assert f'custom_unet.{key}' in _raises(ValueError, **{key: bad}), (key, bad)
        try: CustomUNet(**{key: bad}); raise AssertionError((key, bad))
        except ValueError as e: assert key in str(e)
assert "'attention'" in _raises(ValueError, bottleneck='aspp,attention')                    # strings already behaved

In [ ]:
# test: base_channels / max_channels defaults are named once and used by the class, the validator and the factory (#97)
import inspect, json as _json
_params = inspect.signature(CustomUNet).parameters
assert _params['base_channels'].default == _DEFAULT_BASE_CHANNELS == 32 and _params['max_channels'].default == _DEFAULT_MAX_CHANNELS == 512
_nb_path = next(p for p in (Path.cwd() / '09_model.ipynb', Path.cwd() / 'nbs' / '09_model.ipynb') if p.exists())
exported_text = '\n'.join(''.join(c['source']) for c in _json.load(open(_nb_path))['cells']
                         if c['cell_type'] == 'code' and ''.join(c['source']).startswith('#| export'))
assert "get('base_channels', 32)" not in exported_text and "get('max_channels', 512)" not in exported_text

In [ ]:
# test: one table drives every bottleneck; valid configs build exactly the same networks as before (#97)
import hashlib
pinned = {   # (bottleneck, block_style): (parameter count, md5 of the state_dict keys), recorded before the refactor
    ('none', 'double'): (121042, '75b40017'), ('none', 'residual'): (126426, '1ef8ecee'),
    ('aspp', 'double'): (164322, 'ed0816a0'), ('aspp', 'residual'): (169706, 'b32528b6'),
    ('attention', 'double'): (137810, 'f4bf4c33'), ('attention', 'residual'): (143194, '82eee1e9'),
}
for (bn, bs), (n_params, key_hash) in pinned.items():
    m = CustomUNet(1, 2, 3, base_channels=8, bottleneck=bn, block_style=bs)
    assert (sum(p.numel() for p in m.parameters()), hashlib.md5('|'.join(m.state_dict()).encode()).hexdigest()[:8]) == (n_params, key_hash), (bn, bs)
assert "== 'aspp'" not in exported_text and "== 'attention'" not in exported_text          # no per-name branches left outside the table

# a new bottleneck is one table row: the class, the config validation and the factory all pick it up
_row_built = []
_BOTTLENECKS['twice'] = (lambda ch, knobs, norm, groups: (_row_built.append(ch), nn.Identity())[1], lambda knobs, ch, size, prefix: knobs)
try:
    assert isinstance(CustomUNet(1, 1, 3, 8, bottleneck='twice').bottleneck, nn.Identity) and _row_built == [64]
    _fm = get_model_from_cfg(OmegaConf.merge(cu_cfg, {'model': {'custom_unet': {'bottleneck': 'twice', 'base_channels': 8, 'encoder_depth': 3}}}), 1, 32)
    assert isinstance(_fm.bottleneck, nn.Identity) and _row_built == [64, 64]
finally:
    del _BOTTLENECKS['twice']
assert "'twice'" in _raises(ValueError, bottleneck='twice')                                # gone again after the row is removed